#### This is the roadmap we will be using for this notebook

1. Problem statement
2. Objectives / research questions
3. Data and sources
4. Data cleaning
5. Analysis (charts and statistics)
6. Model (projects only)
7. Findings
8. Key actions / recommendations
9. Limitations and next steps
10. References

### 1. Problem statement

#### The Official Picture: The Low-Unemployment Paradox
Nigeria’s headline unemployment rate appears remarkably low on paper—the National Bureau of Statistics (NBS) reported an official rate of **4.3%** for the second quarter of 2024. However, this metric masks critical realities of the Nigerian labor market:
- Under standard international definitions, an individual is classified as *unemployed* only if they have **zero work** in the reference week and are actively seeking work.
- Anyone engaged in even an hour of economic activity counts as *employed*, regardless of whether the work pays or yields a living income.
- Most employment in Nigeria is informal: the NBS estimates informal employment at approximately **93%**, with only **12% to 13%** of employed workers holding formal wage jobs between late 2022 and 2023.
- For young people aged 15 to 24, unemployment was 6.5% in Q2 2024, and the broader NBS measure that adds time-related underemployment to unemployment was about 13% (NBS, Q2 2024; see references 1 and 2).
- Under this framework, a roadside hawker who works 60 hours a week and earns almost nothing still counts as fully employed.
- While the NBS also reports **underemployment**, its metric is strictly *time-related*—defined as working under 40 hours per week while being willing and available to work more. It fails to capture people who work long hours for very little or no pay.

---

#### 2. Empirical Evidence from Microdata: The Unpaid Labor Reality
Our preliminary analysis of individual-level survey microdata highlights the scale of this distortion:
- Across each of the three quarters studied (Q3 2024, Q1 2025, and Q2 2025), between **25% and 28%** of employed young Nigerians aged 15 to 34 reported receiving **zero payment** for their work (raw counts, to be recalculated with survey weights).
- These individuals are predominantly unpaid family workers helping in agricultural plots or small household micro-enterprises.
- Although officially counted as employed, they derive no direct financial compensation, leaving them highly vulnerable.

---

#### 3. The Core Problem: Policy Blindspots
The current reporting framework creates serious blindspots for policy and economic planning:
- Policymakers can see how many young Nigerians have *no work at all*, but remain blind to how many are working without enough hours or adequate pay.
- They also lack granular visibility into **who** these underutilised youth are, **where** they live (across states, geopolitical zones, and rural vs. urban settings), and what structural barriers they face.
- Consequently, public interventions—including vocational training programs, subsidized credit/loans, and youth employment schemes—risk being aimed at the wrong target populations or misunderstanding the nature of youth distress.

---

#### 4. Project Approach: Broadening the Lens to "Labour Underutilisation"
To address this challenge, this project moves beyond headline employment figures to study **youth labour underutilisation**—defined as people who have no work, or whose work gives them too few hours or too little pay:
- **Three-Tier Classification Framework:** Using individual-level microdata from three waves of the NBS Nigeria Labour Force Survey (NLFS) between July 2024 and June 2025, we sort young Nigerians aged 15 to 34 into three distinct groups:
  1. **Unemployed:** Active searchers with no work and immediate availability.
  2. **Underemployed:** Working, but constrained by time-deficits (< 40 hours wanting more), zero pay, or low earnings (< two-thirds of the quarterly median).
  3. **Adequately Employed:** Working with standard hours and above-threshold earnings.
- **Empirical Profiling:** We analyze which background factors—including education level, sex, age, state of residence, and urban or rural setting—are most strongly linked to each outcome group, enabling targeted policy interventions.
- **Predictive Machine Learning:** We build and evaluate machine learning models (e.g., Logistic Regression, Decision Tree, Random Forest) to test how reliably a young person's employment vulnerability can be predicted from their observable socioeconomic background.

### 2. Objectives/Research questions

**Main objective:** To sort Nigerians aged 15 to 34 into unemployed, underemployed or adequately employed using NBS Labour Force Survey data, to find which background factors are linked to each group, and to test how well a model can predict a person's group.

| # | Research question | How we answer it |
| :-: | :--- | :--- |
| **1** | What share of young Nigerians is in each group, and does it differ by state, sex, education and urban or rural setting? | Weighted percentages with margins of error, in tables and charts |
| **2** | Among the underemployed, why are they underemployed (too few hours, no pay, low pay), and what kind of jobs do they do? | Percentages and charts, plus the job-status mix of each class |
| **3** | Which background factors are most strongly linked to being underemployed rather than adequately employed? | Odds ratios (logistic regression), permutation importance and the model |
| **4** | How well can simple models (Logistic Regression, Decision Tree, Random Forest) predict a young person's group, compared with always guessing the most common group? | The machine learning model |

### 3. Data and sources

We use individual-level microdata from the National Bureau of Statistics (NBS) **Nigeria Labour Force Survey (NLFS)** [4]. The survey samples **35,520 households nationwide** annually, with **8,880 households selected each quarter**. Each row in our data files represents one person living in a surveyed household.

#### Quarters Used

| Quarter | Period | All rows | Youth (15–34) | Status |
| :--- | :--- | :--- | :--- | :--- |
| **Q3 2024** | Jul–Sep 2024 | 43,955 | 13,419 | **Used** |
| **Q4 2024** | Oct–Dec 2024 | n/a | n/a | **Excluded**: The file fails to load and is about 40% smaller than the others, which suggests truncation/corruption |
| **Q1 2025** | Jan–Mar 2025 | 42,385 | 13,013 | **Used** |
| **Q2 2025** | Apr–Jun 2025 | 39,934 | 12,360 | **Used** |

> *Note: "Youth" counts already exclude a small number of "reserve sample" rows (73 in Q3 2024, 97 in Q1 2025, and 66 in Q2 2025).*

---

#### Target Population Breakdown (Unweighted Counts)
*Who ends up in our dataset across the three retained quarters (Q3 2024, Q1 2025, Q2 2025):*

| Group | People | What happens |
| :--- | :--- | :--- |
| **Youth aged 15–34, all three quarters** | **38,792** | **Starting point** |
| • Have a job | 26,142 | Retained: Classes 1 and 2 |
| • No job, searching and available | 1,941 | Retained: Class 0 (unemployed) |
| • No job and not searching (students, homemakers, others) | 10,572 | **Excluded**: Not in the labour force |
| • Searching but not available to start | 137 | **Excluded**: Do not meet standard unemployed definition |

So our modelling data is about **28,000 young people** (26,142 + 1,941 = 28,083), before further cleaning. Class 0 (unemployed) is roughly **7%** of them.

---

#### Target Group Classification (Outcome Variable)
*How each person gets a group (the survey column that answers each question):*

| Class | Label | Defining Rule | Survey Columns Used |
| :---: | :--- | :--- | :--- |
| **0** | **Unemployed** | Has no job; looked for paid work or tried to start a business in the last 4 weeks; and could start now or within two weeks. | `mjj4` blank;<br>`um1_1` / `um1_2` = YES;<br>`um10a` or `um10b` = YES |
| **1** | **Underemployed** | Has a job and satisfies at least one of:<br>**(a) Time-related:** Works under 40 usual hours, wants more, and could start them within two weeks;<br>**(b) Unpaid:** Is paid nothing (unpaid work);<br>**(c) Low pay:** Earns under two-thirds of that quarter's median monthly pay. | `mjj4` filled;<br>**Hours:** `sjj5` (or `mjj12`), `sjj7`, `sjj8`;<br>**Pay:** `sjj10`, `sjj12`, `sjj12a`, `sjj5b`, `sjj5c` |
| **2** | **Adequately employed** | Has a job and none of the underemployment conditions above. | Same columns as Class 1 |

---

#### Features, Survey Weights, and Terms
- **Background factors the model sees (candidate list):** Age (`dc5`), sex (`dc3`), education (`ed6`), state (`id2_state`), urban or rural (`id5_sector`), and quarter. Marital status, disability and household structure are on the proposal's candidate list, but they were not loaded in Step 1, so they are not used in this version.
- **Data Leakage Prevention:** The model never sees the hours, pay, or job-search columns, since those define the groups.
- **Survey weights:** Every row has a weight (`popw`). The sample is about 8,880 households per quarter, far fewer than the whole country, so the weight says how many Nigerians that person stands for. All percentages we report use the weights; without them, over-sampled places would look more important than they are.
- **Access and terms:** Files are the NBS microdata [4]. We use them under the NBS terms and do not redistribute them.

### 4. Data Cleaning

#### Our first goal is to produce one combined dataset containing young Nigerians aged 15–34 from the three selected NLFS quarters.

We'll use:
- Q3 2024
- Q1 2025
- Q2 2025
We will exclude reserve-sample records, preserve the original survey variables, and add a column identifying each quarter.

First, we need a reliable combined dataset so that the classification and machine learning stages use the correct records.


In [ ]:
# STEP 1: LOAD, FILTER, AND COMBINE NLFS DATASETS
# UPDATED VERSION WITH EMPLOYMENT SCREENING QUESTIONS

import pandas as pd
import numpy as np
import zipfile
import os
import glob

from google.colab import files

# 1. DATASET FOLDER
DATA_DIR = "/content/NLFS_Dataset"
os.makedirs(DATA_DIR, exist_ok=True)

# 2. CHECK WHETHER THE DATASETS HAVE ALREADY BEEN EXTRACTED
required_files = {
    "2024q3": "nlfs_2024q3_indiv.dta",
    "2025q1": "nlfs_2025q1_indiv.dta",
    "2025q2": "nlfs_2025q2_indiv.dta"
}

FILES = {}

for quarter, filename in required_files.items():

    matches = glob.glob(
        os.path.join(DATA_DIR, "**", filename),
        recursive=True
    )

    if matches:
        FILES[quarter] = matches[0]

# 3. UPLOAD ZIP ONLY IF DATASETS ARE NOT ALREADY EXTRACTED
if len(FILES) != 3:

    print("Please upload your NLFS Dataset ZIP file.")

    uploaded = files.upload()

    zip_files = [
        name for name in uploaded
        if name.lower().endswith(".zip")
    ]

    if not zip_files:
        raise ValueError("Please upload the original NLFS ZIP file.")

    with zipfile.ZipFile(zip_files[0], "r") as archive:
        archive.extractall(DATA_DIR)

    FILES = {}

    for quarter, filename in required_files.items():

        matches = glob.glob(
            os.path.join(DATA_DIR, "**", filename),
            recursive=True
        )

        if not matches:
            raise FileNotFoundError(f"{filename} was not found.")

        FILES[quarter] = matches[0]

# 4. VALID SURVEY SAMPLES
VALID_SAMPLES = {
    "2024q3": "NLFS target",
    "2025q1": "Original",
    "2025q2": "Original"
}

# 5. REQUIRED COLUMNS
KEEP_COLUMNS = [
    # Personal characteristics
    "dc3", "dc5", "ed6",
    "id2_state", "id5_sector", "popw",
    "sample",

    # Employment screening questions
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b", "abs2", "abs3", "abs4",

    # Agricultural employment questions
    "agf1a_1", "agf1a_2", "agf1a_3", "agf1a_4",
    "agf1b_1", "agf1b_2", "agf1b_3", "agf1b_4",
    "agf2a", "agf2d", "agf4", "agf5", "agf6",

    # Main job and working conditions
    "mjj4", "mjj8b_9", "mjj12",

    # Working hours and underemployment
    "sjj3", "sjj4", "sjj5",
    "sjj5b", "sjj5c", "sjj5d",
    "sjj6", "sjj7", "sjj8", "sjj9",

    # Earnings and payment
    "sjj10", "sjj12", "sjj12a",

    # Job search and availability
    "um1_1", "um1_2",
    "um10a", "um10b"
]

# Remove accidental duplicate column names
KEEP_COLUMNS = list(dict.fromkeys(KEEP_COLUMNS))

# 6. LOAD AND FILTER EACH QUARTER
quarter_data = []

for quarter, file_path in FILES.items():

    print(f"\nProcessing {quarter}...")

    df = pd.read_stata(
        file_path,
        convert_categoricals=True
    )

    # Filter the correct survey sample
    sample_values = df["sample"].astype("string").str.strip()

    df = df.loc[
        sample_values == VALID_SAMPLES[quarter]
    ].copy()

    # Keep youth aged 15–34
    age = pd.to_numeric(
        df["dc5"].astype("string"),
        errors="coerce"
    )

    df = df.loc[age.between(15, 34)].copy()

    # Retain required columns
    df = df.reindex(columns=KEEP_COLUMNS)

    # Standardise categorical data types
    for col in df.select_dtypes(include="category").columns:
        df[col] = df[col].astype("string")

    df["quarter"] = quarter

    quarter_data.append(df)

    print(f"{quarter}: {len(df):,} youth records")

# 7. COMBINE ALL QUARTERS
youth_data = pd.concat(
    quarter_data,
    ignore_index=True
)

# 8. DISPLAY RESULTS
print("\nCOMBINED DATASET SHAPE:", youth_data.shape)

print("\nRECORDS BY QUARTER:")
print(youth_data["quarter"].value_counts().sort_index())

print("\nNEW EMPLOYMENT SCREENING COLUMNS:")
print([
    col for col in KEEP_COLUMNS
    if col.startswith(("atw", "abs", "agf"))
])

print("\nFIRST FIVE RECORDS:")
display(youth_data.head())

print("\nUPDATED DATASET LOADED SUCCESSFULLY!")

#### Check column names and duplicates

In [ ]:

# Display the names of all columns in our combined dataset.
# This allows us to confirm which variables are available
# before we begin cleaning or transforming their values.

print("Columns in the combined dataset:")
print(youth_data.columns.tolist())

# Check whether any column name appears more than once.
# Duplicate names can cause confusion when selecting variables,
# so we want to confirm that each column has a unique name.

duplicate_columns = youth_data.columns[
    youth_data.columns.duplicated()
]

print("\nNumber of duplicate columns:", len(duplicate_columns))


### Calculate the percentage of missing values for each column separately for each survey quarter.

In [ ]:
# This helps us detect variables that may be unavailable,
# unanswered, or recorded differently across quarters.
#
# Important: Some missing responses are intentional because
# the survey asks certain questions only to eligible respondents.
# We should not automatically replace every missing value.

missing_by_quarter = (
    youth_data.groupby("quarter")
    .apply(
        lambda df: df.isna().mean() * 100,
        include_groups=False
    )
    .round(1)
)

# Display missing-value percentages for the employment
# and earnings variables most relevant to our classification.

columns_to_check = [
    "mjj4", "sjj5", "sjj7", "sjj8",
    "sjj10", "sjj12", "sjj12a",
    "um1_1", "um10a", "um10b"
]

display(missing_by_quarter[columns_to_check].T)


<!Doctype html>
<html>
    <body>
        <img src=IMAGES/missing_values.png>
    </body>
</html>



### **Why are so many values missing?**
- Consider sjj5, which records weekly working hours.

> A young person who has no job would not normally have working hours to report.
>
>Similarly, um1_1 asks about job searching. It is mainly relevant to people without employment.
>
>This is called skip logic: respondents answer different questions depending on their employment circumstances and previous responses.
>
>So a missing value can mean the question did not apply, rather than the data was collected incorrectly.


#### Examine how respondents are distributed across the different main-job arrangements recorded in the survey.

In [ ]:
# We use a cross-tabulation to compare the response categories
# across Q3 2024, Q1 2025, and Q2 2025.
#
# Including dropna=False ensures that respondents without
# a recorded main-job arrangement are also counted.

job_arrangements = pd.crosstab(
    youth_data["mjj4"],
    youth_data["quarter"],
    dropna=False
)

display(job_arrangements)


#### These variables help identify young people who are not employed but are actively looking for work and available to start working.

In [ ]:
# The survey uses separate questions for job searching and
# availability, so we inspect their responses before using
# them to construct the unemployment classification.

search_variables = [
    "um1_1",
    "um1_2",
    "um10a",
    "um10b"
]

# Display each variable separately to make the results
# easier to interpret and compare across survey quarters.

for variable in search_variables:

    print(f"\nResponses for {variable}:")

    response_table = pd.crosstab(
        youth_data[variable],
        youth_data["quarter"],
        dropna=False
    )

    display(response_table)


##### Observation: We have now confirmed that the main-job and job-search responses can be compared across all three quarters. This brings us closer to creating our employment classifications.

<!Doctype html>
<html>
    <body>
        <img src=IMAGES/farming_observation.png>
    </body>
</html>

### Identify respondents with a recorded job

##### Create a Boolean variable that identifies respondents who have a recorded main-job arrangement in the NLFS dataset.

In [ ]:
# The mjj4 variable describes the respondent's main employment
# arrangement, including wage employment, self-employment,
# apprenticeships, and helping in household businesses.
#
# For now, we treat a non-missing mjj4 response as evidence
# that the respondent has a job. This is a preliminary indicator;
# we will validate it against the original employment questions
# before using it as a final research classification.

youth_data["has_job"] = youth_data["mjj4"].notna()

# Display the number of respondents with and without a
# recorded job, separately for each survey quarter.

job_status_table = pd.crosstab(
    youth_data["quarter"],
    youth_data["has_job"]
)

display(job_status_table)


### Identify active job searchers

In [ ]:
# Identify respondents who reported looking for paid work
# OR attempting to establish their own business.
#
# We use the OR operator (|) because answering YES to either
# question is sufficient evidence of active job searching.
#
# Missing values are not interpreted as YES. We also restrict
# the indicator to respondents without a recorded job, since
# the unemployment classification concerns people without work.

youth_data["searching"] = (
    (~youth_data["has_job"])
    & (
        youth_data["um1_1"].eq("YES").fillna(False)
        | youth_data["um1_2"].eq("YES").fillna(False)
    )
)

# Count job searchers in each quarter to check whether
# the results agree with our earlier diagnostic findings.

search_table = pd.crosstab(
    youth_data["quarter"],
    youth_data["searching"]
)

display(search_table)


### Identify unemployed respondents

##### Determine whether respondents who are actively searching for employment are also available to begin working.

In [ ]:
# The NLFS first asks whether a respondent could have started
# work during the previous week (um10a). If the answer is NO,
# the survey may ask whether they could start within two weeks
# (um10b).
#
# We consider a respondent available when either question
# contains a YES response. This follows the operational rule
# established for our project.

youth_data["available"] = (
    youth_data["um10a"].eq("YES").fillna(False)
    | youth_data["um10b"].eq("YES").fillna(False)
)

# A respondent is classified as unemployed when they:
# 1. Have no recorded job;
# 2. Are actively searching for employment or business work;
# 3. Are available to begin working.

youth_data["unemployed"] = (
    ~youth_data["has_job"]
    & youth_data["searching"]
    & youth_data["available"]
)

# Display unemployment counts by survey quarter.
# These are sample counts, not population unemployment rates.

unemployment_table = (
    youth_data.groupby("quarter")["unemployed"]
    .sum()
    .astype(int)
)

display(unemployment_table)


<!Doctype html>
<html>
    <body>
        <img src=IMAGES/searching_vs_unemployed.png>
    </body>
</html>

*These are unweighted sample counts.*

> The difference between 2,078 job searchers and 1,941 unemployed respondents is important: **137 people were searching but did not satisfy our availability condition.**

---

### **Next: Preparing the underemployment classification**
*We've established who is unemployed. Now we need to identify young people who have jobs but experience inadequate employment.*

Our project uses three conditions:
1. **Insufficient working hours:** Working fewer than 40 hours per week, wanting additional hours, and being available for them.

2. **Unpaid employment:** Having a job but reporting no payment.

3. **Low earnings:** Earning below two-thirds of the quarter-specific median monthly earnings.
A person meeting at least one of these conditions will be classified as underemployed under our project's operational definition.

> `These are research-defined criteria, not the official NBS underemployment definition.`

### **Examine working hours**

In [ ]:

# STEP 5A: Examine weekly working hours among employed youth.
#
# The sjj5 variable records the total number of hours a
# respondent usually works per week across their main job
# and any secondary job.
#
# We focus only on respondents identified as having a job,
# because working-hours underemployment is a condition
# that applies to employed individuals.
#
# This step does not classify anyone as underemployed yet.
# It simply helps us understand the distribution of working
# hours and identify missing or potentially invalid values.

employed_youth = youth_data.loc[
    youth_data["has_job"]
].copy()

# Convert working hours to numeric values.
# Any values that cannot be interpreted as numbers are
# converted to NaN rather than causing the code to fail.

employed_youth["weekly_hours"] = pd.to_numeric(
    employed_youth["sjj5"],
    errors="coerce"
)

# Display summary statistics for working hours by quarter.
# The median is particularly useful because it is less
# sensitive to extreme values than the mean.

hours_summary = (
    employed_youth.groupby("quarter")["weekly_hours"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
)

display(hours_summary)


### **Examine desire and availability for additional hours**

In [ ]:
# Investigate whether employed respondents want
# additional working hours and whether they are available
# to work those additional hours.
#
# sjj7 records whether the respondent wants more hours.
# sjj8 records whether the respondent is available for
# additional hours, subject to the questionnaire's routing.
#
# Both conditions matter because working fewer than
# 40 hours alone does not necessarily indicate
# involuntary time-related underemployment.

for variable in ["sjj7", "sjj8"]:

    print(f"\nResponses for {variable}:")

    # Compare responses across the three survey quarters,
    # including missing values so that questionnaire
    # routing remains visible in our results.

    response_table = pd.crosstab(
        employed_youth[variable],
        employed_youth["quarter"],
        dropna=False
    )

    display(response_table)



<!Doctype html>
<html>
    <body>
        <img src=IMAGES/time_related_indicator_entry.png>
    </body>
</html>


The median is **42 hours per week in all three quarters**. This means half of the employed respondents worked 42 hours or fewer, and approximately half worked 42 hours or more.

### **Construct the time-related underemployment indicator**

In [ ]:

# STEP 5C: Create the time-related underemployment indicator.
#
# Our project defines time-related underemployment as a
# situation where an employed young person:
#
# 1. Usually works fewer than 40 hours per week;
# 2. Wants to work additional hours; AND
# 3. Is available to work those additional hours.
#
# We require all three conditions because working fewer
# than 40 hours does not necessarily mean someone is
# underemployed. For example, a person may voluntarily
# choose part-time employment.
#
# The '&' operator represents logical AND, meaning
# every condition must be satisfied simultaneously.

youth_data["weekly_hours"] = pd.to_numeric(
    youth_data["sjj5"],
    errors="coerce"
)

youth_data["time_underemployed"] = (
    youth_data["has_job"]
    & youth_data["weekly_hours"].lt(40)
    & youth_data["sjj7"].eq("YES").fillna(False)
    & youth_data["sjj8"].eq("YES").fillna(False)
)

# Count how many employed respondents satisfy all three
# conditions, separately for each survey quarter.
#
# These are unweighted counts of survey respondents,
# not national estimates of underemployment prevalence.

time_underemployment_counts = (
    youth_data.groupby("quarter")["time_underemployed"]
    .sum()
    .astype(int)
)

display(time_underemployment_counts)


### **Validate the classification**

In [ ]:

# STEP 5D: Check whether our time-underemployment
# classification follows the intended rules.
#
# Every respondent classified as time-underemployed must:
# - Have a recorded job;
# - Work fewer than 40 hours;
# - Want additional working hours;
# - Be available for additional working hours.
#
# These assertions help catch accidental classification
# errors before we proceed to the next indicator.

classified = youth_data.loc[
    youth_data["time_underemployed"]
]

assert classified["has_job"].all(), (
    "Some classified respondents have no recorded job."
)

assert classified["weekly_hours"].lt(40).all(), (
    "Some classified respondents work 40 hours or more."
)

assert classified["sjj7"].eq("YES").all(), (
    "Some classified respondents do not want more hours."
)

assert classified["sjj8"].eq("YES").all(), (
    "Some classified respondents are not available."
)

print("Time-related underemployment validation passed.")

# Display the number and percentage of employed youth
# meeting our time-underemployment definition.
#
# The denominator is employed respondents, rather than
# all young people, because time-related underemployment
# applies only to people who have jobs.

time_summary = (
    youth_data.loc[youth_data["has_job"]]
    .groupby("quarter")["time_underemployed"]
    .agg(["sum", "count"])
)

time_summary["percentage"] = (
    time_summary["sum"] / time_summary["count"] * 100
).round(2)

display(time_summary)


<!Doctype html>
<html>
    <body>
        <img src=IMAGES/time-related-indicator.png>
    </body>
</html>

***These are unweighted sample percentages.***

> This means approximately **14 out of every 100 employed young people** in our pooled sample satisfy all three conditions: working fewer than 40 hours, wanting more hours, and being available for them.

### **Next: Identifying unpaid employment**

> ***Remember that our project considers unpaid employment another dimension of labour underutilisation.***

- For example, imagine a young Nigerian who works 45 hours weekly in a family business but receives no payment.

- That person would not satisfy our time-underemployment condition because they work more than 40 hours.

- However, our broader research definition would still identify them as underutilised because they receive no payment.

**This is why we are building separate indicators.**

> **Note:** Still under underemployment

### **Identify respondents reporting no payment**

In [ ]:

# STEP 6A: Construct the unpaid-employment indicator.
#
# The sjj10 variable records how respondents report
# their earnings, including the period of payment
# (hourly, daily, weekly, monthly, or annually).
#
# Importantly, the variable also contains a response
# category called "NO PAYMENT".
#
# We use this explicit response to identify unpaid workers.
# We do NOT interpret missing earnings amounts as unpaid
# employment, because missing values may result from
# questionnaire routing or nonresponse.
#
# Our indicator applies only to employed respondents,
# since unpaid employment is one of the conditions
# used to evaluate the adequacy of existing work.

youth_data["unpaid_employment"] = (
    youth_data["has_job"]
    & youth_data["sjj10"].eq("NO PAYMENT").fillna(False)
)

# Count unpaid employed respondents in each quarter.
# These counts describe survey records and are not yet
# weighted estimates for Nigeria's youth population.

unpaid_counts = (
    youth_data.groupby("quarter")["unpaid_employment"]
    .sum()
    .astype(int)
)

display(unpaid_counts)


### **Validate unpaid employment and calculate percentages**

In [ ]:

# STEP 6B: Validate the unpaid-employment classification.
#
# Every respondent classified as unpaid must:
# 1. Have a recorded job;
# 2. Have explicitly selected "NO PAYMENT" in sjj10.
#
# These checks protect us from accidentally labelling
# respondents with missing earnings as unpaid workers.

unpaid_workers = youth_data.loc[
    youth_data["unpaid_employment"]
]

assert unpaid_workers["has_job"].all(), (
    "Some unpaid respondents have no recorded job."
)

assert unpaid_workers["sjj10"].eq("NO PAYMENT").all(), (
    "Some unpaid respondents did not report NO PAYMENT."
)

print("Unpaid-employment validation passed.")

# Calculate the proportion of employed youth who
# explicitly reported receiving no payment.
#
# We divide by the number of employed respondents
# within each quarter, rather than all youth records.

unpaid_summary = (
    youth_data.loc[youth_data["has_job"]]
    .groupby("quarter")["unpaid_employment"]
    .agg(["sum", "count"])
)

unpaid_summary["percentage"] = (
    unpaid_summary["sum"] / unpaid_summary["count"] * 100
).round(2)

display(unpaid_summary)


<!Doctype html>
<html>
    <body>
        <img src=IMAGES/low_pay_underemployment_indicator.png>
    </body>
</html>

> ***These are unweighted sample results.***

This means approximately **26 out of every 100 employed youth** in our pooled dataset explicitly reported receiving no payment.

- An important distinction: this is our project's unpaid-employment indicator. It does not automatically mean that all these respondents are in `illegal employment` or that they receive no economic benefit whatsoever.
---

### **Preparing the low-earnings indicator**

**This is the most technically sensitive part of our classification because earnings were recorded differently across the survey quarters.**

**`Our eventual goal`** is to identify employed young people earning less than two-thirds of their quarter's median monthly earnings.

> ***But before we calculate that threshold, we must establish how the original earnings responses are stored.***

### **Examine earnings reporting periods**

In [ ]:

# STEP 7A: Inspect how employed respondents report their earnings.
#
# The sjj10 variable records the period associated with
# a respondent's reported earnings. Examples include
# DAILY, WEEKLY, MONTHLY, and ANNUALLY.
#
# These periods are important because earnings cannot be
# compared fairly until they are expressed in the same unit.
#
# For example, earning ₦10,000 per day is very different
# from earning ₦10,000 per month.
#
# We also retain the NO PAYMENT category so that unpaid
# workers remain distinguishable from paid workers.

earnings_period_table = pd.crosstab(
    youth_data.loc[youth_data["has_job"], "sjj10"],
    youth_data.loc[youth_data["has_job"], "quarter"],
    dropna=False
)

display(earnings_period_table)


### **Check exact earnings and earnings bands**

In [ ]:

# STEP 7B: Examine the structure of the earnings variables.
#
# Our earlier diagnostics showed that Q3 2024 records
# earnings in ranges (bands), while the 2025 datasets
# contain exact amounts for some respondents.
#
# This cell examines sjj12 and sjj12a separately for
# each quarter to confirm the formats currently stored
# in our combined DataFrame.
#
# We inspect only a small sample of non-missing values
# rather than printing thousands of individual earnings.
#
# No earnings are converted or estimated at this stage.
# This is important because we want to avoid introducing
# unsupported assumptions into our low-pay classification.

for quarter in ["2024q3", "2025q1", "2025q2"]:

    quarter_workers = youth_data.loc[
        (youth_data["quarter"] == quarter)
        & youth_data["has_job"]
    ]

    print(f"\n{'=' * 45}")
    print(f"EARNINGS STRUCTURE: {quarter}")
    print(f"{'=' * 45}")

    for variable in ["sjj12", "sjj12a"]:

        values = quarter_workers[variable].dropna()

        print(f"\nVariable: {variable}")
        print(f"Non-missing responses: {len(values):,}")
        print(f"Data type: {quarter_workers[variable].dtype}")

        print("Example recorded values:")
        print(values.head(10).tolist())


<!Doctype html>
<html>
    <body>
        <img src=IMAGES/earning.png>
    </body>
</html>

> ***The 19,247 paid workers have earnings information, but not all of it has the same precision.***

For instance, a person earning ₦10,000 weekly and someone earning ₦10,000 monthly should not be treated as having the same monthly earnings.

**We need to convert their amounts into a common unit.**


### **How we'll handle the earnings bands**

<!Doctype html>
<html>
    <body>
        <img src=IMAGES/Earning_bands.png>
    </body>
</html>


**The final category is open-ended. For now, we'll use ₦500,000 as a conservative lower-bound approximation, but we'll mark it as estimated and test its effect later.**

### **Create an earnings-band conversion dictionary**

In [ ]:

# STEP 7C: Define approximate earnings values for survey bands.
#
# Some NLFS respondents report their earnings as a range
# rather than giving an exact monetary amount.
#
# To make these responses usable in numerical analysis,
# we assign each closed earnings band its midpoint.
#
# For example, the ₦30,000–₦50,000 band is represented
# by ₦40,000, the midpoint of that interval.
#
# The lowest band is approximated using ₦2,500.
# The highest band is open-ended, so we provisionally
# represent it using its lower boundary of ₦500,000.
#
# These values are estimates, not respondents' actual
# reported earnings. We will preserve that distinction
# and test the sensitivity of our results later.

earnings_band_midpoints = {
    "< 5K": 2500,
    ">=5K - <10k": 7500,
    ">=10K - <20k": 15000,
    ">=20K - <30k": 25000,
    ">=30K - <50k": 40000,
    ">=50K - <70k": 60000,
    ">=70K - <100k": 85000,
    ">=100K - <500k": 300000,
    ">=500K": 500000
}

print("Earnings-band mapping created successfully.")


### **Extract exact earnings and estimated band values**

In [ ]:
import html

# STEP 7D: Harmonize the two earnings-reporting formats.
#
# Q3 2024 stores paid respondents' earnings as bands
# in sjj12. In the 2025 quarters, sjj12 usually contains
# exact amounts, while sjj12a provides bands for respondents
# who did not report an exact amount.
#
# First, we convert HTML-encoded characters back into
# their ordinary symbols so that the band labels match
# our conversion dictionary.
#
# We then create a single earnings variable containing:
# - Exact amounts when available;
# - Estimated band values otherwise.
#
# We also record whether each amount is exact or estimated,
# which will help us evaluate uncertainty later.

def normalize_earnings_text(value):
    if pd.isna(value):
        return pd.NA
    return html.unescape(str(value)).strip()


youth_data["sjj12_clean"] = youth_data["sjj12"].map(
    normalize_earnings_text
)

youth_data["sjj12a_clean"] = youth_data["sjj12a"].map(
    normalize_earnings_text
)

# Convert sjj12 to a number where it contains an exact
# monetary amount. Band labels and non-numeric responses
# automatically become missing values.

exact_earnings = pd.to_numeric(
    youth_data["sjj12_clean"],
    errors="coerce"
)

# Find the band estimate in either survey variable.
# The second mapping handles the supplementary band
# responses found in the 2025 datasets.

band_from_sjj12 = youth_data["sjj12_clean"].map(
    earnings_band_midpoints
)

band_from_sjj12a = youth_data["sjj12a_clean"].map(
    earnings_band_midpoints
)

estimated_earnings = band_from_sjj12.combine_first(
    band_from_sjj12a
)

# Prefer an exact monetary amount whenever one exists.
# Otherwise, use the corresponding band estimate.

youth_data["reported_earnings"] = (
    exact_earnings.combine_first(estimated_earnings)
)

# Track the source of each amount so that estimated
# earnings are never confused with exact observations.

youth_data["earnings_source"] = np.select(
    [
        exact_earnings.notna(),
        estimated_earnings.notna()
    ],
    [
        "exact",
        "band_estimate"
    ],
    default="missing"
)

print("Earnings formats harmonized successfully.")


### **Validate the earnings conversion**

In [ ]:
# STEP 7E: Verify that the harmonization process
# retained the expected number of earnings records.
#
# We count exact earnings, estimated earnings bands,
# and missing values separately for each survey quarter.
#
# This is especially important because the three
# quarters use different earnings-reporting formats.
#
# Unexpected counts could indicate that some band
# labels were not recognized by our mapping dictionary.

earnings_validation = pd.crosstab(
    youth_data["quarter"],
    youth_data["earnings_source"]
)

display(earnings_validation)

# Identify paid respondents whose earnings could not
# be converted into either an exact or estimated amount.
#
# We expect this count to be zero based on the earlier
# diagnostic results, but we verify it explicitly.

unmatched_paid = youth_data.loc[
    youth_data["has_job"]
    & youth_data["sjj10"].ne("NO PAYMENT")
    & youth_data["reported_earnings"].isna()
]

print(
    "\nPaid respondents with unmatched earnings:",
    len(unmatched_paid)
)


### **Next: Convert earnings to monthly equivalents**

**We now need to make earnings comparable.**

For example, `₦10,000 earned daily is not equivalent to ₦10,000 earned monthly.`

***We'll convert all reported earnings to approximate monthly amounts using the payment period recorded in `sjj10`.***

### **Prepare the payment-period conversion factors**

In [ ]:

# STEP 7F: Define how each earnings-reporting period
# will be converted into an approximate monthly amount.
#
# Respondents report earnings using different periods:
# hourly, daily, weekly, fortnightly, monthly, or annually.
#
# We use the survey's reported work schedule where
# available, rather than assuming every person works
# the same number of hours or days.
#
# For weekly earnings, we use 52 weeks / 12 months.
# For fortnightly earnings, we use 26 periods / 12 months.
# For annual earnings, we divide by 12.
#
# Daily earnings require reported working days per week.
# Hourly earnings require reported weekly working hours.
#
# These conversions create comparable approximations,
# not necessarily exact monthly take-home income.

weeks_per_month = 52 / 12
fortnights_per_month = 26 / 12

print("Weeks per month:", round(weeks_per_month, 3))
print("Fortnights per month:", round(fortnights_per_month, 3))


### **Calculate monthly earnings**

In [ ]:

# STEP 7G: Convert earnings to monthly equivalents.
#
# The reported_earnings variable contains either an
# exact amount or an estimate from an earnings band.
#
# The sjj10 variable identifies the reporting period.
# We apply a different conversion formula depending
# on that period.
#
# Daily earnings are multiplied by the reported number
# of working days per week and the average weeks per month.
#
# Hourly earnings are multiplied by the reported number
# of working hours per week and weeks per month.
#
# Respondents with no valid schedule information will
# retain missing monthly earnings rather than receiving
# an unsupported default assumption.

youth_data["workdays_per_week"] = pd.to_numeric(
    youth_data["sjj5b"], errors="coerce"
)

youth_data["monthly_earnings"] = np.nan

period = youth_data["sjj10"]
amount = youth_data["reported_earnings"]

# Respondents who report monthly earnings already
# have values expressed in the desired time unit.

youth_data.loc[
    period.eq("MONTHLY"), "monthly_earnings"
] = amount

# Convert weekly earnings using the average number
# of weeks in a calendar month.

youth_data.loc[
    period.eq("WEEKLY"), "monthly_earnings"
] = amount * weeks_per_month

# Convert fortnightly earnings using the average
# number of two-week periods per calendar month.

youth_data.loc[
    period.eq("FORTNIGHTLY (EVERY TWO WEEKS)"),
    "monthly_earnings"
] = amount * fortnights_per_month

# Convert annual earnings into a monthly average.

youth_data.loc[
    period.eq("ANNUALLY"), "monthly_earnings"
] = amount / 12

# Convert daily earnings using the respondent's
# reported number of working days per week.

youth_data.loc[
    period.eq("DAILY"), "monthly_earnings"
] = (
    amount
    * youth_data["workdays_per_week"]
    * weeks_per_month
)

# Convert hourly earnings using the respondent's
# reported total weekly working hours.

youth_data.loc[
    period.eq("HOURLY"), "monthly_earnings"
] = (
    amount
    * youth_data["weekly_hours"]
    * weeks_per_month
)

print("Monthly earnings conversion completed.")


### **Validate monthly earnings coverage**

In [ ]:

# STEP 7H: Check whether paid respondents received
# a valid monthly earnings estimate.
#
# This validation helps identify respondents whose
# earnings could not be converted because their
# working schedule was missing or invalid.
#
# We also display the median monthly earnings by
# quarter as a preliminary diagnostic.
#
# These medians are not yet our final low-pay thresholds,
# because suspicious earnings values still need review.

paid_workers = youth_data.loc[
    youth_data["has_job"]
    & youth_data["sjj10"].ne("NO PAYMENT")
]

monthly_coverage = (
    paid_workers.groupby("quarter")["monthly_earnings"]
    .agg(["count", "median", "min", "max"])
    .round(2)
)

display(monthly_coverage)

missing_monthly = paid_workers[
    paid_workers["monthly_earnings"].isna()
]

print(
    "\nPaid respondents without monthly earnings:",
    len(missing_monthly)
)


> ***These are preliminary, unweighted estimates based on the earnings-conversion assumptions.***

**The medians look plausible enough to investigate further, but the extreme values require attention.**

**For example, ₦300 million in monthly earnings is unusually high, while ₦4.33 is extremely low.**

<!Doctype html>
<html>
    <body>
        <img src=IMAGES/low-pay_metric.png>
    </body>
</html>

### **Flag suspicious earnings**

In [ ]:

# STEP 8A: Identify potentially unreliable monthly earnings.
#
# Some survey respondents have unusually small or large
# calculated monthly earnings.
#
# These values may result from data-entry errors,
# incorrect reporting periods, or genuine outliers.
#
# Rather than immediately deleting or replacing them,
# we create separate flags so that we can investigate
# their frequency and decide how to handle them.
#
# The values ₦1,000 and ₦5,000,000 are preliminary
# diagnostic thresholds, not official statistical
# definitions of valid or invalid earnings.

youth_data["earnings_too_low"] = (
    youth_data["monthly_earnings"].notna()
    & youth_data["monthly_earnings"].lt(1000)
)

youth_data["earnings_too_high"] = (
    youth_data["monthly_earnings"].notna()
    & youth_data["monthly_earnings"].gt(5_000_000)
)

# Count suspicious earnings values by quarter.
# This allows us to determine whether unusual amounts
# are concentrated in a particular survey period.

earnings_flags = (
    youth_data.groupby("quarter")[
        ["earnings_too_low", "earnings_too_high"]
    ]
    .sum()
)

display(earnings_flags)


### **Examine the suspicious records**

In [ ]:

# STEP 8B: Inspect respondents with unusual earnings.
#
# We display the original reported amount, payment
# period, work schedule, and calculated monthly amount.
#
# Comparing these fields helps us understand whether
# an unusual monthly value originates from the survey
# response itself or from our conversion formula.
#
# We display only a small sample to keep the notebook
# readable. No observations are modified here.

suspicious_earnings = youth_data.loc[
    youth_data["earnings_too_low"]
    | youth_data["earnings_too_high"],
    [
        "quarter",
        "sjj10",
        "reported_earnings",
        "earnings_source",
        "weekly_hours",
        "workdays_per_week",
        "monthly_earnings"
    ]
]

print(
    "Total suspicious earnings records:",
    len(suspicious_earnings)
)

display(
    suspicious_earnings.sort_values(
        "monthly_earnings"
    ).head(15)
)

display(
    suspicious_earnings.sort_values(
        "monthly_earnings",
        ascending=False
    ).head(15)
)


<!Doctype html>
<html>
    <body>
        <h3><b>The flagged report</b></h3>
        <img src=IMAGES/flagged_records.png>
    </body>
</html>

**These 305 records represent approximately 1.58% of the 19,247 paid workers.**

What makes these records suspicious?
Our screenshots show examples such as:
- ₦10 reported as monthly earnings.
- ₦40 reported as annual earnings.
- ₦300 million reported as monthly earnings.
- Respondents reporting hourly earnings but zero weekly working hours.
These values deserve investigation, but they are not automatically errors.

For example, a respondent reporting zero weekly hours may have been temporarily absent from work. Multiplying their hourly earnings by zero would produce ₦0, even though that doesn't necessarily mean their normal monthly earnings are zero.

*This reveals a limitation in our conversion method, not necessarily a problem with the respondent's answer.*

### **Create a reviewed earnings variable**

In [ ]:
# STEP 8C: Create a separate earnings variable for
# quality-controlled analysis.
#
# We preserve monthly_earnings as the original converted
# value so that no information is permanently overwritten.
#
# The new variable, monthly_earnings_reviewed, initially
# contains the same values as monthly_earnings.
#
# We temporarily mark values below ₦1,000 or above
# ₦5,000,000 as missing in the reviewed variable.
#
# These cutoffs are provisional diagnostic rules,
# NOT official NBS thresholds or proof that the
# observations are incorrect.
#
# Importantly, we do not delete the respondents.
# Their demographic and employment information remains
# available for other parts of our analysis.

youth_data["monthly_earnings_reviewed"] = (
    youth_data["monthly_earnings"].copy()
)

# Identify values that require additional scrutiny.
# We reuse the diagnostic flags created in Cell 8A.

questionable_earnings = (
    youth_data["earnings_too_low"]
    | youth_data["earnings_too_high"]
)

# Temporarily exclude questionable amounts from the
# reviewed earnings measure, without modifying the
# original monthly_earnings column.

youth_data.loc[
    questionable_earnings,
    "monthly_earnings_reviewed"
] = np.nan

print(
    "Earnings values set aside for review:",
    questionable_earnings.sum()
)

print(
    "Original earnings column preserved:",
    "monthly_earnings" in youth_data.columns
)


### **Compare original and reviewed medians**

In [ ]:

# STEP 8D: Compare median monthly earnings before
# and after setting aside questionable values.
#
# The median represents the middle earnings value
# when respondents are ordered from lowest to highest.
#
# Unlike the mean, the median is relatively resistant
# to unusually large earnings values.
#
# Comparing these two versions helps us understand
# whether the provisional data-quality rule materially
# changes our estimate of typical earnings.
#
# We restrict this comparison to paid respondents,
# because unpaid employment is already measured
# separately in our classification.

paid_mask = (
    youth_data["has_job"]
    & youth_data["sjj10"].ne("NO PAYMENT")
)

median_comparison = (
    youth_data.loc[paid_mask]
    .groupby("quarter")[
        ["monthly_earnings", "monthly_earnings_reviewed"]
    ]
    .median()
    .round(2)
)

median_comparison.columns = [
    "original_median",
    "reviewed_median"
]

# Calculate the difference between the two medians.
# A small difference would suggest that the median
# is relatively stable despite the flagged values.

median_comparison["difference"] = (
    median_comparison["reviewed_median"]
    - median_comparison["original_median"]
).round(2)

display(median_comparison)


### **Check how many earnings values remain**

In [ ]:

# STEP 8E: Verify how many paid respondents retain
# usable monthly earnings after provisional screening.
#
# This check allows us to document the effect of
# the screening rule transparently.
#
# The respondents are not removed from youth_data.
# Only their reviewed earnings amounts are missing.
#
# Later, we can compare classification results
# with and without this screening decision.

earnings_coverage = (
    youth_data.loc[paid_mask]
    .groupby("quarter")
    .agg(
        paid_workers=("monthly_earnings", "count"),
        reviewed_earnings=(
            "monthly_earnings_reviewed", "count"
        )
    )
)

earnings_coverage["set_aside"] = (
    earnings_coverage["paid_workers"]
    - earnings_coverage["reviewed_earnings"]
)

display(earnings_coverage)


The Q3 2024 median remained unchanged. The Q1 2025 median increased slightly, while the Q2 2025 median increased by approximately 5.5%.

This suggests that our low-pay classification could be affected by the earnings-screening decision, particularly in Q2 2025.

### **Calculate the thresholds**

<!Doctype html>
<html>
    <body>
        <h4><b>The metric for calculating low pay threshold</b></h4>
        <img src=IMAGES/low-pay_calculation_metric.png>
    </body>
</html>

In [ ]:

# STEP 9A: Calculate a quarter-specific low-pay threshold.
#
# Our project defines low pay as monthly earnings below
# two-thirds of the median monthly earnings among paid
# young workers in the same survey quarter.
#
# We calculate separate thresholds for each quarter
# because typical earnings may differ across periods.
#
# The calculation uses monthly_earnings_reviewed,
# which temporarily excludes the 305 values flagged
# for further investigation.
#
# This is our project's operational definition.
# It is not an official NBS low-pay threshold.
#
# These medians are currently unweighted and provisional.

quarter_medians = (
    youth_data.loc[paid_mask]
    .groupby("quarter")["monthly_earnings_reviewed"]
    .median()
)

low_pay_thresholds = quarter_medians * (2 / 3)

# Display the median and corresponding threshold
# for each survey quarter.

threshold_table = pd.DataFrame({
    "median_monthly_earnings": quarter_medians,
    "low_pay_threshold": low_pay_thresholds
}).round(2)

display(threshold_table)


### **Assign thresholds to individual respondents**

In [ ]:

# STEP 9B: Attach the correct quarter-specific low-pay
# threshold to every respondent in the combined dataset.
#
# The map() method matches each person's quarter with
# the corresponding threshold calculated in Cell 9A.
#
# For example, someone surveyed in Q3 2024 receives
# the Q3 2024 threshold, rather than the Q1 2025
# or Q2 2025 threshold.
#
# This prevents us from comparing respondents against
# a threshold calculated for a different survey period.

youth_data["low_pay_threshold"] = (
    youth_data["quarter"].map(low_pay_thresholds)
)

# Create a preliminary low-pay indicator.
#
# A respondent is low-paid when:
# 1. They have a recorded job;
# 2. They are not classified as unpaid;
# 3. Their reviewed monthly earnings are available;
# 4. Their monthly earnings fall below their quarter's
#    low-pay threshold.
#
# Respondents with unavailable reviewed earnings are
# left unclassified for low pay, rather than being
# automatically labelled as adequately paid.

low_pay_known = (
    youth_data["has_job"]
    & ~youth_data["unpaid_employment"]
    & youth_data["monthly_earnings_reviewed"].notna()
)

youth_data["low_paid"] = pd.Series(
    pd.NA,
    index=youth_data.index,
    dtype="boolean"
)

youth_data.loc[low_pay_known, "low_paid"] = (
    youth_data.loc[low_pay_known, "monthly_earnings_reviewed"]
    < youth_data.loc[low_pay_known, "low_pay_threshold"]
)

print("Low-pay indicator created successfully.")


### **Validate the low-pay indicator**

In [ ]:

# STEP 9C: Summarize low-pay classifications by quarter.
#
# We count three outcomes:
# - True: respondent is classified as low-paid;
# - False: respondent has usable earnings and is not low-paid;
# - <NA>: low-pay status cannot currently be determined.
#
# We focus on paid employed respondents because unpaid
# employment is already handled by a separate indicator.
#
# Keeping unknown cases visible is important:
# otherwise, missing earnings could accidentally be
# interpreted as evidence of adequate employment.

paid_low_pay = youth_data.loc[
    youth_data["has_job"]
    & ~youth_data["unpaid_employment"]
].copy()

low_pay_summary = pd.crosstab(
    paid_low_pay["quarter"],
    paid_low_pay["low_paid"].astype("string"),
    dropna=False
)

display(low_pay_summary)

# Confirm that respondents with reviewed earnings
# excluded from the analysis remain unclassified.

assert youth_data.loc[
    paid_mask
    & youth_data["monthly_earnings_reviewed"].isna(),
    "low_paid"
].isna().all()

print("\nLow-pay missing-value validation passed.")


<!Doctype html>
<html>
    <body>
        <h3><b>Low-pay classification</b></h3>
        <img src=IMAGES/low-pay_classification.png>
    </body>
</html>

**Importantly, unknown does not mean adequately paid. We must preserve that distinction when constructing our final target variable.**

---

### **Next: Check overlap between our three indicators**
**Some young people may satisfy multiple underemployment conditions.**

> - For example, someone could work 25 hours weekly, want additional hours, and also earn below the low-pay threshold.
>
> - We must count that person only once in the final underemployed category.
>
> - Before creating the final classification, let's inspect the overlap

### **Summarize the three indicators**

In [ ]:
# STEP 10A: Summarize the three conditions used in our
# research-defined labour-underutilisation classification.
#
# We have already constructed:
# 1. time_underemployed: insufficient hours, desire for
#    additional hours, and availability for more work.
# 2. unpaid_employment: explicitly reported no payment.
# 3. low_paid: monthly earnings below two-thirds of the
#    quarter-specific median.
#
# We display the number of employed respondents meeting
# each condition separately for every survey quarter.
#
# These counts may overlap, so they must not be added
# together to obtain the total underemployed population.

indicator_summary = (
    youth_data.loc[youth_data["has_job"]]
    .groupby("quarter")
    .agg(
        time_underemployed=("time_underemployed", "sum"),
        unpaid_employment=("unpaid_employment", "sum"),
        low_paid=("low_paid", "sum")
    )
)

display(indicator_summary)


### **Identify overlapping conditions**

In [ ]:

# STEP 10B: Examine how many underemployment conditions
# each employed respondent satisfies.
#
# A respondent may satisfy zero, one, two, or three
# conditions simultaneously.
#
# Because low_paid contains missing values for the
# 305 earnings records requiring review, we temporarily
# treat unknown low-pay status as False ONLY when counting
# already-confirmed conditions.
#
# This does not mean those respondents are adequately paid.
# Their unresolved low-pay status will be handled explicitly
# when constructing the final employment classification.

employed_mask = youth_data["has_job"]

youth_data["confirmed_conditions"] = (
    youth_data["time_underemployed"].astype(int)
    + youth_data["unpaid_employment"].astype(int)
    + youth_data["low_paid"].fillna(False).astype(int)
)

# Display how many employed respondents satisfy each
# possible number of confirmed underemployment conditions.
#
# This helps us identify overlap and prevents double-counting
# when constructing the final target variable.

overlap_table = pd.crosstab(
    youth_data.loc[employed_mask, "confirmed_conditions"],
    youth_data.loc[employed_mask, "quarter"]
)

display(overlap_table)


### **Examine unresolved low-pay cases**

In [ ]:
# STEP 10C: Investigate the 305 employed respondents whose
# low-pay status remains unknown after earnings screening.
#
# Some of these respondents may already qualify as
# underemployed because they satisfy the working-hours
# condition.
#
# Others may satisfy no confirmed condition, meaning
# we cannot yet decide whether they are adequately employed.
#
# We separate these groups so that missing earnings do
# not lead to incorrect final employment labels.

unknown_low_pay = youth_data.loc[
    youth_data["has_job"]
    & youth_data["low_paid"].isna()
].copy()

unknown_summary = (
    unknown_low_pay.groupby("quarter")
    .agg(
        total_unknown=("low_paid", "size"),
        already_time_underemployed=(
            "time_underemployed", "sum"
        )
    )
)

unknown_summary["still_unresolved"] = (
    unknown_summary["total_unknown"]
    - unknown_summary["already_time_underemployed"]
)

display(unknown_summary)


**An important issue in the unknown earnings count**

The output of STEP 10C shows:
Unknown low-pay status ---> 7,200

Already time-underemployed ---> 972

Reported as still unresolved ---> 6,228


*But we previously established that only 305 paid workers had earnings set aside.*

Why are we now seeing 7,200 unknown cases?

**Because our low_paid variable intentionally contains missing values for unpaid workers as well as paid workers with screened earnings.**

***The 7,200 consists of:***
- 6,895 unpaid workers, whose low-pay status is not applicable because they're already classified through the unpaid-employment condition.
- 305 paid workers whose earnings were flagged for review.
Therefore, the 6,228 figure is not the number of genuinely unresolved employment classifications. It includes unpaid workers who are already confirmed as underemployed.

> **We need to correct that before proceeding.**

### **Identify confirmed underemployment**

In [ ]:
# STEP 11A: Identify employed respondents who satisfy
# at least one confirmed underemployment condition.
#
# Our project's broader underemployment definition
# includes three independent conditions:
#
# 1. Working fewer than 40 hours per week while wanting
#    and being available for additional working hours.
#
# 2. Explicitly reporting NO PAYMENT for employment.
#
# 3. Having reviewed monthly earnings below two-thirds
#    of the median for the corresponding survey quarter.
#
# We combine these conditions using logical OR (|).
# This ensures that respondents satisfying multiple
# conditions are counted only once.
#
# Missing low-pay values are temporarily treated as
# False here ONLY because we're identifying people
# who already satisfy at least one confirmed condition.
# Unknown cases will be handled separately.

youth_data["confirmed_underemployed"] = (
    youth_data["has_job"]
    & (
        youth_data["time_underemployed"]
        | youth_data["unpaid_employment"]
        | youth_data["low_paid"].fillna(False)
    )
)

# Count respondents who meet at least one confirmed
# underemployment condition in each survey quarter.

confirmed_counts = (
    youth_data.groupby("quarter")["confirmed_underemployed"]
    .sum()
    .astype(int)
)

display(confirmed_counts)


### **Identify genuinely unresolved cases**

In [ ]:

# STEP 11B: Identify employed respondents whose final
# employment category cannot yet be determined.
#
# A respondent is genuinely unresolved only when:
#
# 1. They have a recorded job;
# 2. They do not satisfy any confirmed underemployment
#    condition; AND
# 3. Their low-pay status is unknown.
#
# This excludes unpaid workers, because their unpaid
# employment already establishes underemployment.
#
# It also excludes people whose working-hours condition
# establishes underemployment despite unknown earnings.
#
# We retain unresolved respondents in the dataset,
# but will not automatically classify them as adequately
# employed in the machine learning target.

youth_data["employment_unresolved"] = (
    youth_data["has_job"]
    & ~youth_data["confirmed_underemployed"]
    & youth_data["low_paid"].isna()
)

unresolved_counts = (
    youth_data.groupby("quarter")["employment_unresolved"]
    .sum()
    .astype(int)
)

display(unresolved_counts)


### **Validate the separation**

In [ ]:
# STEP 11C: Validate the separation between confirmed
# underemployment and unresolved employment status.
#
# No respondent should simultaneously be classified
# as confirmed underemployed and unresolved.
#
# Every unresolved respondent should have a recorded
# job and an unknown low-pay status.
#
# We also check that the unresolved count cannot
# exceed the 305 paid workers whose earnings were
# previously set aside for review.

assert not (
    youth_data["confirmed_underemployed"]
    & youth_data["employment_unresolved"]
).any(), "Confirmed and unresolved categories overlap."

assert youth_data.loc[
    youth_data["employment_unresolved"], "has_job"
].all(), "An unresolved respondent has no recorded job."

assert youth_data.loc[
    youth_data["employment_unresolved"], "low_paid"
].isna().all(), "An unresolved respondent has known low-pay status."

assert youth_data["employment_unresolved"].sum() <= 305, (
    "Unresolved count exceeds the screened paid-worker count."
)

print("Employment classification separation validated.")

# Display the total number of confirmed underemployed
# and unresolved respondents for each quarter.

classification_check = (
    youth_data.groupby("quarter")[
        ["confirmed_underemployed", "employment_unresolved"]
    ]
    .sum()
)

display(classification_check)


<!Doctype html>
<html>
    <body>
        <h3><b>The unemployment rate</b></h3>
        <img src=IMAGES/unemployment.png>
    </body>
</html>

**This means:**
 - 15,265 employed youth satisfy at least one of our three underemployment conditions.

 - 244 employed youth cannot yet be classified confidently because their earnings were flagged and they satisfy no other confirmed condition.

 - Of the original 305 paid workers with flagged earnings, 61 already satisfy the time-related underemployment condition.


**The 244 unresolved respondents will remain in our analytical dataset but will not receive a three-class machine learning label.**



### **Construct our three-class target variable**
We're now ready to bring together the employment-status and underemployment indicators we've built.

**Our target categories will be:**


|Code| |Category|  |Meaning|

> 0	Unemployed	No recorded job, searching, and available

> 1	Underemployed	Employed and satisfies at least one confirmed underemployment condition

> 2	Adequately employed	Employed, satisfies none of the conditions, and has a known low-pay status

***These categories are specific to our project's operational definition, not official NBS employment categories.***

### **Create the target variable**

In [ ]:

# STEP 12A: Construct the three-class employment target.
#
# Machine learning requires a target variable that tells
# the model which outcome it is learning to predict.
#
# We use the following numerical labels:
#
# 0 = Unemployed
# 1 = Underemployed (our research-defined category)
# 2 = Adequately employed (our research-defined category)
#
# We begin with missing values because not every youth
# respondent belongs to one of these three categories.
#
# In particular, we do not assign a class to:
# - People without recorded employment who do not meet
#   our unemployment criteria;
# - Employed people whose employment adequacy remains
#   unresolved because of uncertain earnings.
#
# This prevents the model from learning unsupported labels.

youth_data["employment_class"] = pd.Series(
    pd.NA,
    index=youth_data.index,
    dtype="Int64"
)

# CLASS 0: Unemployed respondents.
# These respondents have no recorded job, are actively
# searching for work, and are available to start.

youth_data.loc[
    youth_data["unemployed"],
    "employment_class"
] = 0

# CLASS 1: Underemployed respondents.
# These respondents have a recorded job and meet at
# least one confirmed underemployment condition.

youth_data.loc[
    youth_data["confirmed_underemployed"],
    "employment_class"
] = 1

# CLASS 2: Adequately employed respondents.
# These respondents have a recorded job, satisfy no
# confirmed underemployment condition, and have no
# unresolved employment-classification issue.

adequately_employed = (
    youth_data["has_job"]
    & ~youth_data["confirmed_underemployed"]
    & ~youth_data["employment_unresolved"]
)

youth_data.loc[
    adequately_employed,
    "employment_class"
] = 2

print("Three-class employment target created successfully.")


### **Summarize the target categories**

In [ ]:
# STEP 12B: Examine the distribution of our three
# employment categories across the survey quarters.
#
# We include missing values because they represent
# respondents who are outside the modelling population
# or whose employment classification remains unresolved.
#
# This allows us to confirm that no observations were
# silently discarded during target construction.
#
# These are unweighted sample counts.

class_names = {
    0: "Unemployed",
    1: "Underemployed",
    2: "Adequately employed"
}

youth_data["employment_label"] = (
    youth_data["employment_class"].map(class_names)
)

class_distribution = pd.crosstab(
    youth_data["quarter"],
    youth_data["employment_label"],
    dropna=False
)

display(class_distribution)

# Count the total number of respondents who have
# a valid three-class label and can therefore be
# considered for the machine learning dataset.

model_eligible = youth_data[
    youth_data["employment_class"].notna()
]

print(
    "\nRespondents with valid target labels:",
    f"{len(model_eligible):,}"
)


### **Validate the target variable**

In [ ]:

# STEP 12C: Validate the final target assignments.
#
# We check that each class follows the rules defined
# in our methodology and that no respondent receives
# more than one employment category.
#
# We also confirm that respondents with unresolved
# employment status remain unlabelled.
#
# These checks help us catch mistakes before using
# employment_class as the target for machine learning.

assert youth_data.loc[
    youth_data["employment_class"].eq(0).fillna(False),
    "unemployed"
].all(), "Class 0 contains someone not classified as unemployed."

assert youth_data.loc[
    youth_data["employment_class"].eq(1).fillna(False),
    "confirmed_underemployed"
].all(), "Class 1 contains someone not confirmed underemployed."

assert not youth_data.loc[
    youth_data["employment_class"].eq(2).fillna(False),
    "confirmed_underemployed"
].any(), "Class 2 contains an underemployed respondent."

assert youth_data.loc[
    youth_data["employment_unresolved"],
    "employment_class"
].isna().all(), "Unresolved respondents received a target label."

assert set(
    youth_data["employment_class"].dropna().unique()
).issubset({0, 1, 2}), "Unexpected employment class detected."

print("Three-class employment target validation passed.")


### **Validate employment status against the original survey**

In [ ]:
youth_data["has_job"] = youth_data["mjj4"].notna()

**This produced 26,142 employed respondents.**

However, mjj4 describes someone's main-job arrangement. We should independently verify whether the original employment-screening questions support that classification.

### **Check the relevant columns directly in the source files**

In [ ]:

# STEP 13B (CORRECTED): Identify employment-screening
# variables in the original NLFS datasets.
#
# Our combined youth_data DataFrame contains only the
# survey variables selected during data preparation.
# Some original employment-screening variables were
# excluded at that stage.
#
# We now inspect the original Stata files to identify
# variables associated with employment and temporary
# absence from work.
#
# Setting iterator=True creates a StataReader rather
# than immediately loading every respondent's data.
# The reader's variable_labels() method provides a
# dictionary of variable names and their descriptions.
#
# This allows us to inspect the survey metadata
# without changing any of our existing classifications.

import pandas as pd

for quarter, file_path in FILES.items():

    with pd.read_stata(
        file_path,
        iterator=True,
        convert_categoricals=False
    ) as reader:

        # Retrieve variable names and descriptions from
        # the original Stata file's metadata.

        variable_labels = reader.variable_labels()

        # Search for variable names or descriptions
        # containing employment-related keywords.
        #
        # Descriptions are useful because the NLFS
        # often uses abbreviated variable names.

        matching_variables = {
            name: description
            for name, description in variable_labels.items()
            if (
                "atw" in name.lower()
                or "abs" in name.lower()
                or "work" in description.lower()
                or "employment" in description.lower()
            )
        }

        print(f"\n{'=' * 50}")
        print(f"EMPLOYMENT VARIABLES: {quarter}")
        print(f"{'=' * 50}")

        for name, description in matching_variables.items():
            print(f"{name}: {description}")
            with open('relevant_columns.txt', 'a', encoding='utf-8') as file:
                file.write(f"{name}: {description}\n")


**The important discovery is that mjj4, which we previously used to identify employed respondents, describes the arrangement in someone's main job. It isn't itself the survey's initial employment-screening question.**

**Our file identifies these relevant variables:**

<!Doctype html>
<html>
    <body>
        <h3><b></b></h3>
        <img src=IMAGES/Screening_features.png>
    </body>
</html>

These are more appropriate variables for investigating employment status. However, we must inspect their actual response values before deciding how to combine them, particularly for agriculture and temporary absence.

### **Inspect the actual employment-screening responses**

In [ ]:
# STEP 14A: Load only the original employment-screening
# questions from each NLFS survey file.
#
# We are checking whether respondents reported:
# - Working for pay or running a business;
# - Helping in a household business;
# - Having a job while temporarily absent;
# - Participating in agricultural activities.
#
# These variables were not included in our earlier
# combined youth_data dataset.
#
# Reading selected columns reduces memory usage and
# avoids rebuilding the entire dataset.
#
# IMPORTANT: This cell does not modify youth_data,
# has_job, unemployed, or employment_class.

screening_columns = [
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b",
    "agf1a_1", "agf1a_2", "agf1a_3",
    "agf2a", "agf2d"
]

screening_data = {}

for quarter, file_path in FILES.items():

    # Retrieve the variable names available in this
    # particular survey quarter.

    with pd.read_stata(
        file_path,
        iterator=True,
        convert_categoricals=False
    ) as reader:
        available = set(reader.variable_labels())

    # Select only questions actually present in the file.
    # This prevents an error if a quarter omits a variable.

    selected = [
        col for col in screening_columns
        if col in available
    ]

    # Read selected variables with their response labels,
    # making values such as YES and NO easier to interpret.

    screening_data[quarter] = pd.read_stata(
        file_path,
        columns=selected,
        convert_categoricals=True
    )

    print(
        f"{quarter}: "
        f"{len(screening_data[quarter]):,} source records; "
        f"{len(selected)} screening variables loaded"
    )


### **Display response counts by quarter**

In [ ]:

# STEP 14B: Examine the response categories recorded
# for each employment-screening question.
#
# We display counts separately for every survey quarter.
# This is necessary because the questionnaire or its
# response categories may differ between survey periods.
#
# Missing responses are included explicitly.
# A missing response does not automatically mean NO:
# some questions are asked only when earlier answers
# satisfy particular conditions.
#
# These are counts for the original source files,
# not yet the age-filtered youth population.
#
# We will inspect the results before creating any
# replacement employment-status indicator.

for quarter, df in screening_data.items():

    print(f"\n{'=' * 45}")
    print(f"QUARTER: {quarter}")
    print(f"{'=' * 45}")

    for column in df.columns:

        print(f"\n{column}:")

        counts = df[column].value_counts(
            dropna=False
        )

        display(counts.to_frame("respondents"))


### **Identify available respondent identifiers**


**Why this matters for our machine learning model**

We need respondent identifiers for two reasons.

- First, to independently validate the employment classifications we've already constructed.

- Second, to check whether the same respondent appears in multiple quarters. If that happens, we need to prevent their records from being split between training and testing in a way that produces overly optimistic performance estimates.

In [ ]:

# STEP 15A: Identify respondent and household identifiers.
#
# We need to match the original employment-screening
# answers to the correct respondents in youth_data.
#
# We must NOT assume that row 100 in a source dataset
# corresponds to row 100 in youth_data, because we
# previously filtered the data by age and survey rules.
#
# This cell displays the identifier-related columns
# available in both datasets.
#
# We are only inspecting metadata and column names.
# No rows, employment labels, or previous calculations
# will be modified.

# Display the columns currently available in our
# combined youth dataset.

print("COLUMNS IN youth_data:")
print(youth_data.columns.tolist())

with open("columns_dataset.txt", 'w', encoding='utf-8') as file:
    file.write(str(youth_data.columns.tolist()))
    print("Saved in columns dataset")

# Look for likely household and person identifiers
# in the original source files.
#
# These are only candidate columns. We will verify
# which combination uniquely identifies respondents
# before using it to match records.

identifier_terms = [
    "hhid", "person", "member",
    "indiv", "case", "serial", "id"
]

import json
for quarter, file_path in FILES.items():

    with pd.read_stata(
        file_path,
        iterator=True,
        convert_categoricals=False
    ) as reader:

        source_columns = list(reader.variable_labels())

    candidates = [
        col for col in source_columns
        if any(
            term in col.lower()
            for term in identifier_terms
        )
    ]

    print(f"\n{quarter} identifier candidates:")
    print(candidates)
    with open("identifiers.txt", 'a', encoding='utf-8') as file:
        file.write(f"{quarter}: {candidates}\n")



### **Check identifier uniqueness**

In [ ]:

# STEP 15B-1: Check which identifiers uniquely distinguish
# respondents in the original NLFS survey files.
#
# A valid respondent-level identifier should identify
# each individual record without duplicates.
#
# We first examine interview_id and hhroster_id separately,
# then test whether combining them produces a unique key.
#
# This is important because multiple individuals may belong
# to the same household or interview.
#
# We do not merge anything or change youth_data here.
# The purpose is to establish a safe matching key.

identifier_columns = [
    "interview_id",
    "hhroster_id"
]

for quarter, file_path in FILES.items():

    # Read only the candidate identifier columns
    # to keep memory usage low.

    identifiers = pd.read_stata(
        file_path,
        columns=identifier_columns,
        convert_categoricals=False
    )

    print(f"\n{'=' * 45}")
    print(f"QUARTER: {quarter}")
    print(f"{'=' * 45}")

    print("Total records:", len(identifiers))

    # Count distinct values for each candidate
    # identifier separately.

    for column in identifier_columns:

        print(
            f"Unique {column}:",
            identifiers[column].nunique(
                dropna=False
            )
        )

    # Check whether the combination uniquely
    # identifies every respondent.

    duplicate_pairs = identifiers.duplicated(
        subset=identifier_columns
    ).sum()

    print(
        "Duplicate interview_id + hhroster_id pairs:",
        duplicate_pairs
    )

    # Check for missing identifier values, since
    # missing keys can make matching unreliable.

    print(
        "Records with missing identifier:",
        identifiers[identifier_columns]
        .isna()
        .any(axis=1)
        .sum()
    )


### **Inspect identifier formats**

In [ ]:

# STEP 15B-2: Examine the structure of the identifiers.
#
# Even when identifiers are unique, we need to understand
# whether they represent individual respondents, households,
# or survey interviews.
#
# We display only a few sample values from each quarter.
# This helps us check the data types and formatting
# before attempting to match records.
#
# These identifiers are used only for data validation,
# not as predictive features in our machine learning models.

for quarter, file_path in FILES.items():

    identifiers = pd.read_stata(
        file_path,
        columns=["interview_id", "hhroster_id"],
        convert_categoricals=False
    )

    print(f"\n{quarter} identifier examples:")

    display(identifiers.head(5))


<!Doctype html>
<html>
    <body>
        <h3><b></b></h3>
        <img src=IMAGES/identifier_column.png>
    </body>
</html>


**Two important discoveries:**
- `interview_id` identifies a household interview, which is why multiple respondents share the same value.

- `hhroster_id` identifies a household member within that interview.
Together, interview_id + hhroster_id uniquely identifies each respondent within a quarter.

For our combined dataset, we'll also include quarter in the key.

### **Build a source reference table**

**What this cell does:** reads only the identifier columns and a few screening columns from the original NBS files, for all three quarters, without any filtering. The result is `source_reference`, a clean copy of the source records that we can use to check our own filters. The cell also confirms that `quarter + interview_id + hhroster_id` identifies every person only once.

In [ ]:

# STEP 16A: Build a reference table containing the
# respondent identifiers and employment-screening
# answers from the three original NLFS datasets.
#
# We use quarter + interview_id + hhroster_id as the
# respondent-level identification key.
#
# This reference table is kept separate from youth_data.
# We do not overwrite the employment classifications
# or other variables already created in our notebook.
#
# We also retain age, sex, and main-job arrangement
# to support later checks that records were matched
# to the correct respondents.

source_columns = [
    "interview_id",
    "hhroster_id",
    "dc5",
    "dc3",
    "mjj4",
    "atw1",
    "atw2",
    "atw3",
    "abs1a",
    "abs1b",
    "agf1a_1",
    "agf1a_2",
    "agf1a_3",
    "agf2a",
    "agf2d",
    "sample"
]

source_parts = []

for quarter, file_path in FILES.items():

    # Read only the variables required for respondent
    # identification and employment-status validation.

    part = pd.read_stata(
        file_path,
        columns=source_columns,
        convert_categoricals=True
    )

    # Record the quarter so that respondents from
    # different survey periods remain distinguishable.

    part["quarter"] = quarter

    source_parts.append(part)

# Combine the three reference tables vertically.
# This does not merge respondents across households
# or survey periods.

source_reference = pd.concat(
    source_parts,
    ignore_index=True
)

print(
    "Source reference records:",
    len(source_reference)
)

# Verify that the respondent key remains unique
# after combining the three survey periods.

source_key = [
    "quarter",
    "interview_id",
    "hhroster_id"
]

assert not source_reference.duplicated(
    subset=source_key
).any(), "Duplicate respondent identifiers detected."

print("Combined respondent identifiers validated.")


### **Check whether our original youth filters can be reproduced**

In [ ]:

# STEP 16B: Reproduce the youth sample from the
# original survey reference records.
#
# Our project focuses on respondents aged 15–34.
#
# Earlier, we also excluded reserve-sample records
# during the initial data preparation.
#
# Here, we first examine the sample categories rather
# than guessing which category should be excluded.
#
# This is a diagnostic step. We do not filter or merge
# any records yet.

print("Sample categories by quarter:")

display(
    pd.crosstab(
        source_reference["quarter"],
        source_reference["sample"],
        dropna=False
    )
)

# Check the number of respondents aged 15–34
# before applying any sample-category exclusions.
#
# We use numeric conversion for age to ensure
# the filter behaves consistently across quarters.

source_reference["age_numeric"] = pd.to_numeric(
    source_reference["dc5"],
    errors="coerce"
)

age_check = (
    source_reference.loc[
        source_reference["age_numeric"].between(15, 34)
    ]
    .groupby("quarter")
    .size()
)

print("\nYouth aged 15–34 before sample exclusions:")
display(age_check)


<!Doctype html>
<html>
    <body>
        <h3><b></b></h3>
        <img src=IMAGES/reference.png>
    </body>
</html>

**Notice that 81 youth respondents, not all 236 reserve-sample records, were excluded. That's because the reserve samples also contained respondents outside our 15–34 age range.**


## **Reconstruct the source youth population**
Now we'll apply the verified age and sample filters to source_reference.

We'll then confirm that the resulting records match the 38,792 observations in youth_data.

### **Apply the verified filters**

In [ ]:

# STEP 17A: Reconstruct the original youth sample
# using the same age and sample-selection rules
# applied during initial data preparation.
#
# Our research population consists of young people
# aged 15–34 from the main NLFS sample.
#
# The source files use different labels:
# - Q3 2024: NLFS target
# - Q1 and Q2 2025: Original
#
# We explicitly select these categories instead of
# assuming that every record outside the reserve
# sample belongs to the main survey.
#
# This operation creates a separate DataFrame.
# It does not modify youth_data or its existing labels.

main_sample_labels = {
    "2024q3": "NLFS target",
    "2025q1": "Original",
    "2025q2": "Original"
}

source_reference["main_sample"] = (
    source_reference["sample"]
    == source_reference["quarter"].map(main_sample_labels)
)

source_youth = source_reference.loc[
    source_reference["age_numeric"].between(15, 34)
    & source_reference["main_sample"]
].copy()

print("Reconstructed youth records:", len(source_youth))

display(
    source_youth.groupby("quarter").size()
)


### **Validate the reconstructed sample**

In [ ]:

# STEP 17B: Confirm that our reconstructed source
# youth sample matches the existing combined dataset.
#
# Matching record counts is necessary before attempting
# respondent-level validation, but counts alone do not
# prove that records are aligned.
#
# We will perform additional record-level checks before
# attaching employment-screening information.
#
# If the counts differ, the assertion will stop
# the workflow so that we can investigate.

expected_counts = (
    youth_data.groupby("quarter").size()
)

actual_counts = (
    source_youth.groupby("quarter").size()
)

comparison = pd.DataFrame({
    "existing_youth_data": expected_counts,
    "reconstructed_source": actual_counts
})

comparison["difference"] = (
    comparison["reconstructed_source"]
    - comparison["existing_youth_data"]
)

display(comparison)

assert (comparison["difference"] == 0).all(), (
    "Source youth counts do not match existing data."
)

print("Youth sample counts validated successfully.")



<!Doctype html>
<html>
    <body>
        <h3><b>We've successfully reconstructed the exact youth sample used in our project.</b></h3>
        <img src=IMAGES/sample_reconstruct.png>
    </body>
</html>


**This confirms that our age and sample-selection filters reproduce the original youth counts.**

However, matching counts does not guarantee that individual rows are in the same order. We need one final alignment check before attaching the employment-screening variables.

### **Compare records across datasets**

In [ ]:

# STEP 18A: Verify whether the existing youth dataset
# and reconstructed source dataset have matching records.
#
# Both datasets contain exactly 38,792 youth respondents.
# However, equal record counts do not establish that
# the rows correspond to the same individuals.
#
# We compare three independently retained variables:
# - dc5: respondent's age;
# - dc3: respondent's sex;
# - mjj4: respondent's main-job arrangement.
#
# We compare records within each quarter separately,
# resetting their row indexes for this diagnostic.
#
# Missing values are considered matching only when
# both datasets have a missing value in the same position.
#
# IMPORTANT:
# This is a diagnostic comparison, not a respondent-ID
# match. Even perfect agreement across these variables
# would not establish identity with absolute certainty.
#
# No columns or existing classifications are modified.

alignment_results = []

for quarter in ["2024q3", "2025q1", "2025q2"]:

    existing = (
        youth_data.loc[
            youth_data["quarter"] == quarter,
            ["dc5", "dc3", "mjj4"]
        ]
        .reset_index(drop=True)
    )

    reconstructed = (
        source_youth.loc[
            source_youth["quarter"] == quarter,
            ["dc5", "dc3", "mjj4"]
        ]
        .reset_index(drop=True)
    )

    # Compare the two DataFrames position by position.
    # Converting to strings allows us to compare
    # categorical labels consistently.
    #
    # Missing values are standardized to one marker
    # before comparison.

    existing_compare = (
        existing.astype("string").fillna("<MISSING>")
    )

    reconstructed_compare = (
        reconstructed.astype("string").fillna("<MISSING>")
    )

    row_matches = (
        existing_compare.eq(reconstructed_compare).all(axis=1)
    )

    alignment_results.append({
        "quarter": quarter,
        "records": len(existing),
        "matching_rows": int(row_matches.sum()),
        "mismatched_rows": int((~row_matches).sum())
    })

alignment_table = pd.DataFrame(alignment_results)

display(alignment_table)


### **Check whether the alignment is complete**

In [ ]:

# STEP 18B: Validate the row-alignment diagnostic.
#
# We require every compared record to match across
# age, sex, and main-job arrangement.
#
# If mismatches are found, we will investigate the
# ordering rather than attaching source variables
# to potentially incorrect respondents.
#
# This check does not yet attach respondent identifiers
# or alter the employment_class variable.

assert (
    alignment_table["mismatched_rows"] == 0
).all(), (
    "Row alignment mismatch detected. "
    "Do not attach source identifiers yet."
)

print(
    "Age, sex, and main-job arrangement "
    "match across all reconstructed records."
)


## **Verify alignment and attach employment-screening variables**

**We'll use three small code cells with descriptive comments, as we've been doing.**

### **Check additional survey variables**
Instead of relying on only three matching columns, we'll compare every original survey variable that is also present in youth_data.

In [ ]:

# STEP 19A: Perform a stronger respondent-alignment check.
#
# Our previous validation confirmed that age, sex, and
# main-job arrangement matched in all 38,792 records.
#
# We now compare additional variables shared by the
# original source reference and our existing youth_data.
#
# The goal is to reduce the possibility that two different
# respondents happen to share the same age, sex, and job
# arrangement while occupying the same row position.
#
# We preserve quarter order and reset indexes separately.
# No existing employment labels are changed.

shared_columns = [
    col for col in ["dc5", "dc3", "mjj4"]
    if col in youth_data.columns and col in source_youth.columns
]

alignment_checks = []

for quarter in ["2024q3", "2025q1", "2025q2"]:

    existing = (
        youth_data.loc[
            youth_data["quarter"].eq(quarter),
            shared_columns
        ]
        .reset_index(drop=True)
        .astype("string")
        .fillna("<MISSING>")
    )

    original = (
        source_youth.loc[
            source_youth["quarter"].eq(quarter),
            shared_columns
        ]
        .reset_index(drop=True)
        .astype("string")
        .fillna("<MISSING>")
    )

    matches = existing.eq(original).all(axis=1)

    alignment_checks.append({
        "quarter": quarter,
        "columns_compared": len(shared_columns),
        "mismatched_records": int((~matches).sum())
    })

display(pd.DataFrame(alignment_checks))


### **Attach the original screening questions**

In [ ]:

# STEP 19B: Create a validation dataset containing
# the existing youth records and original employment
# screening responses.
#
# IMPORTANT:
# This is a positional reconstruction, not an ID merge.
# It relies on the matching row order established in
# our previous checks.
#
# We work on a COPY of youth_data to preserve all
# existing variables and employment classifications.
#
# We attach respondent identifiers and screening answers
# within each quarter, preserving the quarter-specific
# ordering.
#
# The new dataset will be used only for validation until
# we establish whether the original classifications agree.

screening_vars = [
    "interview_id", "hhroster_id",
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b",
    "agf1a_1", "agf1a_2", "agf1a_3",
    "agf2a", "agf2d"
]

validation_parts = []

for quarter in ["2024q3", "2025q1", "2025q2"]:

    existing = youth_data.loc[
        youth_data["quarter"].eq(quarter)
    ].reset_index(drop=True).copy()

    source = source_youth.loc[
        source_youth["quarter"].eq(quarter),
        screening_vars
    ].reset_index(drop=True)

    # Stop immediately if the quarter contains
    # different numbers of records.

    assert len(existing) == len(source), (
        f"Record count mismatch for {quarter}"
    )

    # Attach the source screening fields by position.
    # The source and existing data must retain their
    # original row ordering for this to be valid.

    for col in screening_vars:
        existing[col] = source[col].to_numpy()

    validation_parts.append(existing)

validation_data = pd.concat(
    validation_parts,
    ignore_index=True
)

assert len(validation_data) == len(youth_data)

print(
    "Validation dataset created:",
    len(validation_data),
    "records"
)


### **Compare employment-screening responses**

In [ ]:

# STEP 19C: Compare the original employment-screening
# responses against our existing has_job indicator.
#
# The survey includes questions about paid work,
# business activities, household-business assistance,
# and temporary absence from work.
#
# We first create a simple diagnostic flag identifying
# respondents who answered YES to any of these questions.
#
# Agricultural production requires additional conditions,
# so we deliberately leave agricultural variables out
# of this initial flag.
#
# IMPORTANT:
# This is NOT a complete official employment indicator.
# It is only a diagnostic comparison against has_job.

direct_work_questions = [
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b"
]

validation_data["direct_work_evidence"] = (
    validation_data[direct_work_questions]
    .eq("YES")
    .any(axis=1)
)

# Compare respondents with and without direct evidence
# of work against our existing has_job classification.
#
# The off-diagonal cells identify records that deserve
# further investigation.
#
# Missing responses may reflect questionnaire routing,
# so lack of a YES response does not prove unemployment.

comparison_table = pd.crosstab(
    validation_data["has_job"],
    validation_data["direct_work_evidence"],
    margins=True
)

display(comparison_table)

# Show the same comparison separately by survey quarter.
# This helps determine whether discrepancies are
# concentrated in a particular dataset.

for quarter, group in validation_data.groupby("quarter"):

    print(f"\nEmployment comparison: {quarter}")

    display(
        pd.crosstab(
            group["has_job"],
            group["direct_work_evidence"]
        )
    )


### **Identify which questions explain the discrepancies**

In [ ]:

# STEP 20A: Investigate respondents who reported some
# work activity but were not classified as employed
# by our existing main-job arrangement indicator.
#
# Our current has_job variable was constructed using
# whether mjj4 contains a valid response.
#
# The original NLFS questionnaire includes earlier
# questions about paid work, business activity,
# household-business assistance, and temporary absence.
#
# We now count how many of the 2,039 discrepant
# respondents answered YES to each question.
#
# A respondent may answer YES to multiple questions,
# so these counts may overlap.
#
# This is diagnostic analysis only. We do not change
# has_job or employment_class at this stage.

work_without_mjj4 = validation_data.loc[
    ~validation_data["has_job"]
    & validation_data["direct_work_evidence"]
].copy()

questions = [
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b"
]

question_counts = (
    work_without_mjj4[questions]
    .eq("YES")
    .sum()
    .sort_values(ascending=False)
)

print(
    "Respondents with work evidence but no mjj4:",
    len(work_without_mjj4)
)

display(
    question_counts.to_frame("YES responses")
)


### **Examine their current employment classifications**

In [ ]:

# STEP 20B: Examine how the discrepant respondents
# were classified in our existing project.
#
# Some may currently be labelled unemployed, while
# others may be outside our three-class modelling
# population.
#
# This distinction matters because reclassifying
# respondents without examining the survey rules
# could introduce errors into our machine learning
# target variable.
#
# We display their current employment labels and
# search/availability indicators without modifying
# any existing columns.

print("Current employment classifications:")

display(
    work_without_mjj4["employment_label"]
    .fillna("Not classified")
    .value_counts()
)

print("\nCurrent unemployment flags:")

display(
    work_without_mjj4["unemployed"]
    .value_counts(dropna=False)
)

print("\nSearching for work:")

display(
    work_without_mjj4["searching"]
    .value_counts(dropna=False)
)


### **Examine the 25 respondents in the opposite direction**

In [ ]:

# STEP 20C: Investigate respondents with a recorded
# main-job arrangement but no YES response in our
# simplified direct-work screening indicator.
#
# These respondents may be involved in agricultural
# employment or may follow different questionnaire
# pathways.
#
# We inspect their main-job arrangements alongside
# agricultural production and temporary-absence
# responses.
#
# We do not assume that missing screening answers
# mean the respondent did not work.

job_without_direct_evidence = validation_data.loc[
    validation_data["has_job"]
    & ~validation_data["direct_work_evidence"]
].copy()

print(
    "Respondents with mjj4 but no direct YES:",
    len(job_without_direct_evidence)
)

display(
    job_without_direct_evidence[
        [
            "quarter",
            "mjj4",
            "agf2a",
            "agf2d",
            "atw1",
            "atw2",
            "atw3",
            "abs1a",
            "abs1b"
        ]
    ].head(25)
)


## **5. Exploratory Data Analysis (EDA)**

#### **Our goal is to understand what the data tells us about youth unemployment and underemployment in Nigeria.**

<!Doctype html>
<html>
    <body>
        <h3><b>EDA Roadmap.</b></h3>
        <img src=IMAGES/EDA_roadmap.png>
    </body>
</html>

## **Prepare our EDA dataset**

#### **Create a separate dataset for analysis**

In [ ]:

# STEP 21A: Prepare the dataset for exploratory data analysis.
#
# Our combined youth_data contains 38,792 respondents,
# but not everyone belongs to our three employment classes.
#
# We select respondents who have a valid employment_class:
# 0 = Unemployed
# 1 = Underemployed
# 2 = Adequately employed
#
# This creates a separate DataFrame called eda_data.
# We preserve youth_data and all its existing variables.
#
# IMPORTANT:
# These are provisional, research-defined categories.
# The EDA results should not be described as official
# NBS unemployment or underemployment estimates.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

eda_data = youth_data.loc[
    youth_data["employment_class"].notna()
].copy()

# Establish a consistent category order so that
# our tables and charts use the same arrangement.

category_order = [
    "Unemployed",
    "Underemployed",
    "Adequately employed"
]

eda_data["employment_label"] = pd.Categorical(
    eda_data["employment_label"],
    categories=category_order,
    ordered=True
)

print("EDA dataset prepared successfully.")
print("Total respondents:", len(eda_data))


### **Overall employment distribution**

In [ ]:

# STEP 22A: Calculate the number and percentage of
# respondents in each employment category.
#
# We use value_counts() to count respondents and
# calculate their share of the analytical sample.
#
# The percentages are unweighted sample percentages.
# They describe the 27,839 classified respondents,
# not the entire Nigerian youth population.
#
# We will later use survey weights for estimates
# intended to represent the population.

employment_counts = (
    eda_data["employment_label"]
    .value_counts()
    .reindex(category_order)
)

employment_percentages = (
    employment_counts / len(eda_data) * 100
)

employment_summary = pd.DataFrame({
    "Respondents": employment_counts,
    "Percentage": employment_percentages.round(2)
})

display(employment_summary)


In [ ]:

# STEP 22B: Visualize the distribution of employment
# categories using a bar chart.
#
# Bar charts are appropriate for comparing the sizes
# of distinct categories.
#
# We display percentages rather than only raw counts
# so that the relative size of each category is clear.
#
# Each bar is labelled with its percentage, making
# the chart easier to interpret in our final report.
#
# The title explicitly identifies the chart as an
# unweighted sample distribution.

plt.figure(figsize=(9, 5))

ax = sns.barplot(
    x=employment_summary.index,
    y=employment_summary["Percentage"],
    hue=employment_summary.index,
    palette="Set2",
    legend=False
)

# Add percentage labels above each bar.

for bar in ax.patches:
    height = bar.get_height()

    ax.annotate(
        f"{height:.1f}%",
        (bar.get_x() + bar.get_width() / 2, height),
        ha="center",
        va="bottom",
        xytext=(0, 5),
        textcoords="offset points",
        fontsize=11,
        fontweight="bold"
    )

plt.title(
    "Employment Categories Among Nigerian Youth\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Employment Category")
plt.ylabel("Percentage of Classified Respondents (%)")
plt.ylim(0, employment_summary["Percentage"].max() + 10)
plt.tight_layout()
plt.show()


### **Employment categories across survey quarters**

In [ ]:

# STEP 23A: Calculate the employment-category
# distribution separately for each survey quarter.
#
# The three datasets contain different numbers of
# respondents, so comparing raw counts alone could
# be misleading.
#
# pd.crosstab() counts respondents by quarter and
# employment category.
#
# normalize="index" divides each category count by
# the total number of classified respondents in
# that quarter.
#
# Multiplying by 100 converts the resulting shares
# into percentages.
#
# Each quarter should therefore sum to approximately
# 100%, allowing fairer comparisons between periods.

quarter_distribution = (
    pd.crosstab(
        eda_data["quarter"],
        eda_data["employment_label"],
        normalize="index"
    ) * 100
)

quarter_distribution = (
    quarter_distribution
    .reindex(columns=category_order)
    .round(2)
)

display(quarter_distribution)


In [ ]:

# STEP 23B: Visualize employment-category shares
# across the three NLFS survey quarters.
#
# A 100% stacked bar chart shows how the composition
# of employment categories differs between periods.
#
# Each bar represents one survey quarter and totals
# 100% of its classified respondents.
#
# We use the same category order and colors throughout
# to make the chart easier to compare and interpret.
#
# These percentages are unweighted and should not
# be interpreted as official national labour rates.

quarter_labels = {
    "2024q3": "Q3 2024",
    "2025q1": "Q1 2025",
    "2025q2": "Q2 2025"
}

plot_data = quarter_distribution.copy()

plot_data.index = [
    quarter_labels.get(q, q)
    for q in plot_data.index
]

ax = plot_data.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["#E76F51", "#E9C46A", "#2A9D8F"],
    width=0.65
)

# Display the percentage inside each segment.
# Small segments remain visible because we place
# labels at their vertical midpoint.

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        label_type="center",
        fontsize=10
    )

plt.title(
    "Employment Categories Across Survey Quarters\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Survey Quarter")
plt.ylabel("Percentage of Classified Respondents (%)")
plt.xticks(rotation=0)
plt.ylim(0, 100)
plt.legend(
    title="Employment Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)
plt.tight_layout()
plt.show()


### **Employment categories by sex**

In [ ]:

# STEP 24A: Examine the distribution of employment
# categories among male and female youth respondents.
#
# This analysis helps answer our research question
# about demographic differences in labour underutilisation.
#
# We first inspect the recorded sex categories to
# confirm how they appear in our analytical dataset.
#
# We then calculate the percentage of respondents
# within each sex category who are unemployed,
# underemployed, or adequately employed.
#
# normalize="index" is important because the number
# of male and female respondents may differ.
# Each row will total approximately 100%.
#
# These are unweighted descriptive percentages, not
# nationally representative population estimates.

print("Sex categories in our dataset:")

display(
    eda_data["dc3"].value_counts(dropna=False)
)

sex_distribution = (
    pd.crosstab(
        eda_data["dc3"],
        eda_data["employment_label"],
        normalize="index"
    ) * 100
)

sex_distribution = (
    sex_distribution
    .reindex(columns=category_order)
    .round(2)
)

display(sex_distribution)


In [ ]:

# STEP 24B: Create a grouped bar chart comparing
# employment categories across sex groups.
#
# Each employment category has separate bars for
# the recorded sex groups.
#
# A grouped bar chart makes it easier to compare
# the unemployment, underemployment, and adequate
# employment percentages between groups.
#
# The percentages are calculated within each sex
# category, not across the entire dataset.
#
# We use consistent colors and percentage labels
# to improve readability in the final report.

sex_plot = sex_distribution.T

ax = sex_plot.plot(
    kind="bar",
    figsize=(10, 6),
    width=0.75
)

# Display percentage labels above each bar.

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3,
        fontsize=10
    )

plt.title(
    "Employment Categories by Sex Among Nigerian Youth\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Employment Category")
plt.ylabel("Percentage Within Sex Group (%)")
plt.xticks(rotation=0)
plt.legend(title="Sex")
plt.ylim(0, sex_distribution.max().max() + 10)
plt.tight_layout()
plt.show()


### **Employment Categories by Age Group.**

In [ ]:

# STEP 25A: Create age groups for demographic analysis.
#
# Our project defines youth as individuals aged 15–34.
# Instead of examining every individual age separately,
# we group respondents into five-year age intervals.
#
# This makes the charts easier to interpret and allows
# us to compare employment outcomes across different
# stages of early adulthood.
#
# pd.cut() assigns each respondent to an age interval.
# The intervals are defined so that each age belongs
# to exactly one group.
#
# We create the new column only in eda_data.
# Our original youth_data remains unchanged.

age_labels = [
    "15–19",
    "20–24",
    "25–29",
    "30–34"
]

eda_data["age_group"] = pd.cut(
    pd.to_numeric(eda_data["dc5"], errors="coerce"),
    bins=[14, 19, 24, 29, 34],
    labels=age_labels
)

# Display the number of respondents in each age group.
# This helps us understand the sample sizes behind
# the percentages we will visualize.

age_group_counts = (
    eda_data["age_group"]
    .value_counts()
    .reindex(age_labels)
)

display(age_group_counts)


In [ ]:

# STEP 25B: Calculate employment-category percentages
# separately for each age group.
#
# We use a cross-tabulation to count the respondents
# in each employment category within each age group.
#
# normalize="index" converts those counts into shares
# of each age group rather than shares of the entire
# analytical sample.
#
# This is important because the number of respondents
# may differ considerably across age groups.
#
# Each age-group row should total approximately 100%.
# These remain unweighted sample percentages.

age_distribution = (
    pd.crosstab(
        eda_data["age_group"],
        eda_data["employment_label"],
        normalize="index"
    ) * 100
)

age_distribution = (
    age_distribution
    .reindex(
        index=age_labels,
        columns=category_order
    )
    .round(2)
)

display(age_distribution)


In [ ]:

# STEP 26A: Visualize employment categories across
# the four youth age groups.
#
# Each age group contains three bars representing
# unemployment, underemployment, and adequate employment.
#
# This chart helps us identify whether employment
# outcomes differ across age groups.
#
# We compare percentages rather than raw counts,
# since the groups may contain different numbers
# of respondents.
#
# The color scheme matches our previous quarter chart:
# red = unemployed, yellow = underemployed,
# green = adequately employed.

ax = age_distribution.plot(
    kind="bar",
    figsize=(11, 6),
    width=0.8,
    color=["#E76F51", "#E9C46A", "#2A9D8F"]
)

# Add percentage labels above the bars to make
# the differences easier to read in our final report.

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3,
        fontsize=9
    )

plt.title(
    "Employment Categories by Age Group Among Nigerian Youth\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Age Group (Years)")
plt.ylabel("Percentage Within Age Group (%)")
plt.xticks(rotation=0)
plt.ylim(0, age_distribution.max().max() + 10)

plt.legend(
    title="Employment Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


### **Employment categories by education level**

In [ ]:

# STEP 27A: Examine educational attainment among
# respondents in our analytical dataset.
#
# The NLFS education variable (ed6) records the
# educational attainment of respondents.
#
# Before creating charts, we inspect the actual
# response categories and their frequencies.
#
# This helps us avoid inventing educational levels
# or combining categories incorrectly.
#
# Missing education responses are included so we
# can see how many respondents lack this information.
#
# This step only reads the existing eda_data DataFrame.
# It does not modify the original survey variables.

education_counts = (
    eda_data["ed6"]
    .value_counts(dropna=False)
)

print("Educational attainment categories:")
display(education_counts.to_frame("Respondents"))


In [ ]:

# STEP 27B: Calculate the percentage of respondents
# in each employment category by education level.
#
# We calculate percentages WITHIN each education
# category, rather than across the whole dataset.
#
# For example, if a particular education group has
# 1,000 classified respondents and 500 are
# underemployed, its underemployment share is 50%.
#
# This allows us to compare education groups fairly,
# even when their sample sizes are different.
#
# Missing education values are excluded from this
# particular comparison, but remain in eda_data.
#
# These are unweighted descriptive percentages and
# do not establish that education causes any
# particular employment outcome.

education_distribution = (
    pd.crosstab(
        eda_data["ed6"],
        eda_data["employment_label"],
        normalize="index"
    ) * 100
)

education_distribution = (
    education_distribution
    .reindex(columns=category_order)
    .round(2)
)

# Include the number of classified respondents in
# each education group to provide context for the
# percentages, especially for smaller groups.

education_sample_sizes = (
    eda_data.groupby("ed6", observed=True)
    .size()
)

education_summary = education_distribution.copy()

education_summary.insert(
    0,
    "Sample size",
    education_sample_sizes.reindex(
        education_summary.index
    )
)

display(education_summary)


In [ ]:

# STEP 28A: Visualize employment-category shares
# across educational attainment groups.
#
# A horizontal 100% stacked bar chart allows us
# to compare the composition of employment outcomes
# across different education levels.
#
# Each bar represents one education category and
# totals 100% of its classified respondents.
#
# We use the same colors as our earlier charts:
# red = unemployed,
# yellow = underemployed,
# green = adequately employed.
#
# The sample size is included beside each education
# category to make small groups easier to identify.
#
# We preserve the survey's original education labels
# rather than introducing unsupported categories.

education_plot = education_distribution.copy()

education_plot.index = [
    f"{level} (n={education_sample_sizes.loc[level]:,})"
    for level in education_plot.index
]

fig, ax = plt.subplots(
    figsize=(12, max(6, len(education_plot) * 0.55))
)

education_plot.plot(
    kind="barh",
    stacked=True,
    ax=ax,
    color=["#E76F51", "#E9C46A", "#2A9D8F"],
    width=0.75
)

# Add percentage labels only to segments large
# enough to display them without excessive overlap.
#
# Very small segments remain visible in the chart,
# and their exact percentages are available in
# the summary table from the previous cell.

for container in ax.containers:
    labels = [
        f"{bar.get_width():.1f}%"
        if bar.get_width() >= 5
        else ""
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=9
    )

ax.set_title(
    "Employment Categories by Education Level\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Percentage Within Education Group (%)")
ax.set_ylabel("Educational Attainment")
ax.set_xlim(0, 100)

ax.legend(
    title="Employment Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


### **Analyse employment categories by state**

In [ ]:

# STEP 29A: Examine the Nigerian states represented
# in our analytical dataset.
#
# The id2_state variable records each respondent's state.
#
# Before comparing employment outcomes geographically,
# we examine how many classified respondents come from
# each state.
#
# State sample sizes may differ because the NLFS uses
# a survey sampling design rather than collecting an
# equal number of respondents from every state.
#
# These counts describe our analytical sample and
# should not be interpreted as state populations.
#
# This cell does not modify eda_data or youth_data.

state_counts = (
    eda_data["id2_state"]
    .value_counts(dropna=False)
    .sort_index()
)

print("Number of state categories:", eda_data["id2_state"].nunique())

display(
    state_counts.to_frame("Classified respondents")
)


### **Calculate employment percentages by state**

In [ ]:

# STEP 29B: Calculate the employment-category
# distribution within each Nigerian state.
#
# We count the classified respondents in each
# employment category for every state.
#
# normalize="index" converts those counts into
# percentages within each state.
#
# For example, a state with 200 classified respondents,
# of whom 100 are underemployed, will have an
# underemployment share of 50%.
#
# This approach allows us to compare states with
# different sample sizes.
#
# We retain the number of respondents per state
# because estimates from smaller samples may be
# less stable.
#
# These are unweighted descriptive percentages,
# not official state-level employment estimates.

state_distribution = (
    pd.crosstab(
        eda_data["id2_state"],
        eda_data["employment_label"],
        normalize="index"
    ) * 100
)

state_distribution = (
    state_distribution
    .reindex(columns=category_order)
    .round(2)
)

# Calculate the sample size for each state.

state_sample_sizes = (
    eda_data.groupby("id2_state", observed=True)
    .size()
)

# Combine sample sizes and employment percentages
# into a single table for easier interpretation.

state_summary = state_distribution.copy()

state_summary.insert(
    0,
    "Sample size",
    state_sample_sizes.reindex(
        state_summary.index
    )
)

# Sort states by their underemployment percentage,
# from highest to lowest.
#
# This is a descriptive ranking of our sample,
# not a ranking of official state unemployment rates.

state_summary = state_summary.sort_values(
    by="Underemployed",
    ascending=False
)

display(state_summary)


In [ ]:

# STEP 30A: Visualize employment-category shares
# across Nigerian states.
#
# Each horizontal bar represents one state.
# The segments show the percentage of classified
# respondents who are unemployed, underemployed,
# or adequately employed.
#
# Every bar totals 100%, allowing comparisons
# between states with different sample sizes.
#
# States are ordered from highest to lowest
# underemployment share based on our sample.
#
# We include the number of respondents beside
# each state name to provide context.
#
# We preserve the same employment-category colors
# used in our previous EDA charts.

state_plot = state_summary[category_order].copy()

state_plot.index = [
    f"{state} (n={int(state_summary.loc[state, 'Sample size']):,})"
    for state in state_plot.index
]

fig, ax = plt.subplots(
    figsize=(13, max(10, len(state_plot) * 0.38))
)

state_plot.plot(
    kind="barh",
    stacked=True,
    ax=ax,
    color=["#E76F51", "#E9C46A", "#2A9D8F"],
    width=0.8
)

# Add percentage labels only when a segment is
# large enough to display them clearly.
#
# Small segments remain visible, while their
# exact percentages can be found in state_summary.

for container in ax.containers:

    labels = [
        f"{bar.get_width():.0f}%"
        if bar.get_width() >= 10
        else ""
        for bar in container
    ]

    ax.bar_label(
        container,
        labels=labels,
        label_type="center",
        fontsize=8
    )

ax.set_title(
    "Employment Categories Across Nigerian States\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Percentage Within State (%)")
ax.set_ylabel("State")
ax.set_xlim(0, 100)

# Place the state with the highest underemployment
# share at the top of the chart.

ax.invert_yaxis()

ax.legend(
    title="Employment Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


In [ ]:

# STEP 31A: Identify states with the highest and
# lowest underemployment shares in our sample.
#
# We display five states at each end of the
# underemployment distribution.
#
# The rankings are descriptive and unweighted.
# They should not be interpreted as statistically
# established differences between state populations.
#
# We include sample sizes so that readers can
# evaluate whether a state has relatively few
# classified respondents.
#
# The full state_summary table remains available
# for the final project report.

print("Five states with the highest underemployment shares:")

display(
    state_summary[
        ["Sample size", "Underemployed"]
    ].head(5)
)

print("\nFive states with the lowest underemployment shares:")

display(
    state_summary[
        ["Sample size", "Underemployed"]
    ].tail(5).sort_values(
        "Underemployed",
        ascending=True
    )
)


### **Analyse the three underemployment conditions**

In [ ]:

# STEP 32A: Calculate how frequently each underemployment
# condition occurs among employed youth respondents.
#
# Our project uses three conditions to identify broader
# forms of labour underutilisation:
#
# 1. Time-related underemployment: insufficient working
#    hours combined with wanting and being available
#    for additional work.
#
# 2. Unpaid employment: respondents whose reported
#    payment status is NO PAYMENT.
#
# 3. Low earnings: paid respondents earning below
#    two-thirds of the quarter-specific median.
#
# We calculate percentages using all respondents with
# has_job == True as the denominator.
#
# IMPORTANT:
# A respondent can satisfy more than one condition,
# so the percentages must not be added together.
#
# Missing low-pay status is not interpreted as evidence
# that a respondent is adequately paid.
#
# These are unweighted descriptive sample percentages.

employed_data = youth_data.loc[
    youth_data["has_job"]
].copy()

condition_columns = {
    "Insufficient working hours": "time_underemployed",
    "Unpaid employment": "unpaid_employment",
    "Low earnings": "low_paid"
}

condition_results = []

for condition_name, column in condition_columns.items():

    # Count only respondents explicitly classified
    # as satisfying this particular condition.
    #
    # Unknown values are excluded from the numerator,
    # but remain in the employed-sample denominator.

    count = int(
        employed_data[column].eq(True).sum()
    )

    percentage = (
        count / len(employed_data) * 100
    )

    condition_results.append({
        "Condition": condition_name,
        "Respondents": count,
        "Percentage": round(percentage, 2)
    })

condition_summary = pd.DataFrame(condition_results)

display(condition_summary)

print(
    "Total employed youth respondents:",
    f"{len(employed_data):,}"
)


In [ ]:

# STEP 32B: Create a horizontal bar chart showing
# the prevalence of each underemployment condition.
#
# Each bar represents the percentage of employed
# youth who satisfy a particular condition.
#
# This chart helps us compare the relative frequency
# of insufficient hours, unpaid employment, and
# low earnings within our analytical sample.
#
# Because respondents may satisfy multiple conditions,
# the bars represent overlapping groups.
#
# The chart should therefore not be interpreted
# as three mutually exclusive employment categories.

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    condition_summary["Condition"],
    condition_summary["Percentage"],
    color=["#457B9D", "#E9C46A", "#E76F51"]
)

# Display the percentage and respondent count at
# the end of each bar for easier interpretation.

for bar, count, percentage in zip(
    bars,
    condition_summary["Respondents"],
    condition_summary["Percentage"]
):

    ax.text(
        bar.get_width() + 0.5,
        bar.get_y() + bar.get_height() / 2,
        f"{percentage:.1f}% (n={count:,})",
        va="center",
        fontsize=10
    )

ax.set_title(
    "Types of Underemployment Among Employed Nigerian Youth\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Percentage of Employed Respondents (%)")
ax.set_ylabel("Underemployment Condition")

ax.set_xlim(
    0,
    condition_summary["Percentage"].max() + 12
)

ax.invert_yaxis()

plt.tight_layout()
plt.show()


In [ ]:

# STEP 33A: Examine how many underemployment conditions
# each employed respondent satisfies simultaneously.
#
# We use the confirmed_conditions variable created
# earlier in our notebook.
#
# A value of:
# 0 = No confirmed underemployment condition;
# 1 = Exactly one confirmed condition;
# 2 = Exactly two confirmed conditions;
# 3 = All three conditions.
#
# Because unpaid employment and low earnings are
# defined separately, we do not expect many, if any,
# respondents to satisfy all three.
#
# IMPORTANT:
# Zero confirmed conditions does not necessarily mean
# adequate employment. Some respondents still have
# unresolved earnings information.

overlap_summary = (
    employed_data["confirmed_conditions"]
    .value_counts()
    .sort_index()
    .rename_axis("Number of conditions")
    .reset_index(name="Respondents")
)

overlap_summary["Percentage"] = (
    overlap_summary["Respondents"]
    / len(employed_data)
    * 100
).round(2)

display(overlap_summary)


#### **Visualize the overlap**

In [ ]:

# STEP 33B: Visualize the number of confirmed
# underemployment conditions per employed respondent.
#
# Unlike the previous chart, these categories are
# mutually exclusive: each respondent belongs to
# exactly one group based on their condition count.
#
# This helps us distinguish people experiencing one
# form of underemployment from those experiencing
# multiple forms simultaneously.
#
# Respondents with zero confirmed conditions may
# include unresolved cases and must not automatically
# be described as adequately employed.

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    overlap_summary["Number of conditions"].astype(str),
    overlap_summary["Percentage"],
    color="#457B9D"
)

for bar, percentage in zip(
    bars,
    overlap_summary["Percentage"]
):

    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{percentage:.1f}%",
        ha="center",
        fontsize=10,
        fontweight="bold"
    )

ax.set_title(
    "Number of Underemployment Conditions Among Employed Youth\n"
    "(Unweighted Analytical Sample)",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Number of Confirmed Conditions")
ax.set_ylabel("Percentage of Employed Respondents (%)")
ax.set_ylim(
    0,
    overlap_summary["Percentage"].max() + 8
)

plt.tight_layout()
plt.show()


#### **Compare underemployment conditions across quarters**

In [ ]:

# STEP 34A: Compare the prevalence of each
# underemployment condition across survey quarters.
#
# Each percentage is calculated using employed
# respondents from that specific quarter.
#
# This is important because the three quarters
# contain different numbers of employed respondents.
#
# We count respondents with a confirmed True value
# for each condition, then divide by the total
# number of employed respondents in that quarter.
#
# These are unweighted descriptive percentages.
# We do not interpret differences between quarters
# as causal changes or official national trends.

quarter_condition_rows = []

for quarter, group in employed_data.groupby("quarter"):

    for condition_name, column in condition_columns.items():

        percentage = (
            group[column].eq(True).sum()
            / len(group)
            * 100
        )

        quarter_condition_rows.append({
            "Quarter": quarter,
            "Condition": condition_name,
            "Percentage": percentage
        })

quarter_condition_data = pd.DataFrame(
    quarter_condition_rows
)

quarter_condition_table = (
    quarter_condition_data.pivot(
        index="Quarter",
        columns="Condition",
        values="Percentage"
    ).round(2)
)

display(quarter_condition_table)


In [ ]:

# STEP 34B: Create a grouped bar chart comparing
# underemployment conditions across survey periods.
#
# Each survey quarter contains three bars:
# insufficient working hours, unpaid employment,
# and low earnings.
#
# The percentages use employed respondents within
# each quarter as the denominator.
#
# This helps us identify whether certain forms
# of labour underutilisation appear more frequently
# in some survey periods than others.
#
# The groups can overlap, so percentages within
# a quarter do not need to sum to 100%.

quarter_condition_plot = quarter_condition_table.copy()

quarter_condition_plot.index = [
    quarter_labels.get(q, q)
    for q in quarter_condition_plot.index
]

ax = quarter_condition_plot.plot(
    kind="bar",
    figsize=(11, 6),
    width=0.75,
    color=["#457B9D", "#E76F51", "#E9C46A"]
)

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        padding=3,
        fontsize=9
    )

plt.title(
    "Underemployment Conditions Across Survey Quarters\n"
    "(Unweighted Employed Youth Sample)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Survey Quarter")
plt.ylabel("Percentage of Employed Respondents (%)")
plt.xticks(rotation=0)
plt.ylim(
    0,
    quarter_condition_table.max().max() + 8
)

plt.legend(
    title="Underemployment Condition",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


### **Survey-weighted employment analysis**

Why do we need survey weights?
Imagine two states:
- State A has 500 respondents in our dataset.
- State B has 300 respondents.
That doesn't necessarily mean State A has a larger youth population or should contribute more to national estimates.

Survey weights help account for differences in sampling probabilities and survey adjustments.

Our popw column provides these weights. However, because we combined three survey quarters, we must calculate each quarter's estimates separately rather than treating the pooled records as one national cross-section.

In [ ]:

# STEP 35A: Prepare the NLFS population weights
# for survey-weighted descriptive analysis.
#
# The popw variable contains a survey population
# weight assigned to each respondent.
#
# Unlike ordinary sample percentages, weighted
# percentages give respondents different contributions
# according to their survey weights.
#
# We convert popw to a numeric format and create
# a separate column for analysis.
#
# We retain only finite, positive weights when
# calculating weighted proportions.
#
# We do NOT modify employment_class, the original
# survey weights, or our previous EDA results.

eda_data["survey_weight"] = pd.to_numeric(
    eda_data["popw"],
    errors="coerce"
)

# Identify valid survey weights.
# Zero, negative, missing, or infinite weights
# cannot contribute meaningfully to the weighted
# percentage calculations used below.

eda_data["valid_weight"] = (
    eda_data["survey_weight"].notna()
    & np.isfinite(eda_data["survey_weight"])
    & eda_data["survey_weight"].gt(0)
)

# Summarize the number of usable weights
# separately for each survey quarter.

weight_summary = (
    eda_data.groupby("quarter")
    .agg(
        classified_respondents=("employment_class", "size"),
        valid_weights=("valid_weight", "sum")
    )
)

weight_summary["invalid_weights"] = (
    weight_summary["classified_respondents"]
    - weight_summary["valid_weights"]
)

display(weight_summary)


In [ ]:

# STEP 35B: Calculate survey-weighted employment
# percentages within each survey quarter.
#
# For each quarter, we sum the population weights
# belonging to each employment category.
#
# We then divide that sum by the total population
# weight of all classified respondents in the
# same quarter.
#
# Formula:
#
# Weighted percentage =
# (sum of weights in category /
#  sum of weights in quarter) * 100
#
# IMPORTANT:
# The denominator contains only respondents with
# valid three-class labels and valid survey weights.
#
# Therefore, these results describe the weighted
# composition of our CLASSIFIED analytical population,
# not unemployment among all Nigerian youth.
#
# We calculate each quarter separately to avoid
# accidentally combining population totals from
# different survey periods.

weighted_eda = eda_data.loc[
    eda_data["valid_weight"]
].copy()

# Sum respondent weights by quarter and category.

weighted_totals = (
    weighted_eda.groupby(
        ["quarter", "employment_label"],
        observed=True
    )["survey_weight"]
    .sum()
    .unstack(fill_value=0)
)

# Convert each quarter's weighted totals into
# percentages that add up to 100%.

weighted_distribution = (
    weighted_totals.div(
        weighted_totals.sum(axis=1),
        axis=0
    ) * 100
)

weighted_distribution = (
    weighted_distribution
    .reindex(columns=category_order)
    .round(2)
)

display(weighted_distribution)


### **Compare weighted and unweighted results**

In [ ]:

# STEP 35C: Compare the weighted and unweighted
# employment distributions for each survey quarter.
#
# Our earlier EDA charts were based on raw respondent
# counts. The new estimates account for the survey
# population weights.
#
# Comparing the two helps us understand whether
# the sampling design materially changes the
# employment-category percentages.
#
# Large differences are not automatically errors.
# They may reflect unequal sampling probabilities
# or population adjustments.
#
# We retain both versions for transparent reporting.

weighted_vs_unweighted = pd.concat(
    {
        "Unweighted (%)": quarter_distribution,
        "Weighted (%)": weighted_distribution
    },
    axis=1
)

display(weighted_vs_unweighted)


In [ ]:

# STEP 36A: Visualize survey-weighted employment
# category percentages across the three quarters.
#
# This chart uses the population weights supplied
# with the NLFS microdata.
#
# Each quarter is represented by one stacked bar
# showing the weighted composition of our three
# employment categories.
#
# Every bar totals 100%.
#
# The chart remains restricted to respondents with
# valid employment labels, so it must not be
# presented as the official national unemployment
# or underemployment rate.
#
# We use the same colors as the earlier charts
# to make weighted and unweighted results easier
# to compare.

weighted_plot = weighted_distribution.copy()

weighted_plot.index = [
    quarter_labels.get(q, q)
    for q in weighted_plot.index
]

ax = weighted_plot.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["#E76F51", "#E9C46A", "#2A9D8F"],
    width=0.65
)

# Add percentage labels inside each bar segment.

for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.1f%%",
        label_type="center",
        fontsize=10
    )

plt.title(
    "Survey-Weighted Employment Categories by Quarter\n"
    "(Classified Nigerian Youth Sample)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Survey Quarter")
plt.ylabel("Weighted Percentage (%)")
plt.xticks(rotation=0)
plt.ylim(0, 100)

plt.legend(
    title="Employment Category",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.tight_layout()
plt.show()


Key findings:
- Underemployment dominates: Approximately 55–56% of the weighted, classified analytical population falls into our research-defined underemployment category in all three quarters.
- Unemployment increases slightly: From 7.3% in Q3 2024 to 8.3% in Q2 2025, although we cannot conclude that this is a statistically significant increase.
- Adequate employment remains relatively stable: Around 36–37% across the three quarters.

### **Weighted results with margins of error (answers research question 1)**

The charts above count respondents. They show who is in **our sample**, not who is in **Nigeria**. In this part we use the survey weights (`popw`) so that each person counts for the number of young Nigerians they represent.

**How we do it (in simple English):**
- Inside each quarter, the weights are scaled to add up to 1. Every quarter then counts equally, and we never add three surveys together as if they were one.
- For each group (for example, women or rural youth), the weighted share is: weight of people in the class ÷ weight of all classified people in that group.
- The **± margin** is a guide to how uncertain a percentage is. It uses the *effective sample size*, which shrinks when the weights are very unequal. It does **not** model the full survey design (clusters and strata), so the true margin is probably a little wider.

**Education** has many detailed codes, so we first group them into a few levels (the proposal asks for this).

#### **Group the detailed education codes into levels**

**What this cell does:** turns the many detailed `ed6` codes (for example `SS3`, `HND1,HND2`, `QURANIC`) into six education levels plus *Unknown* and *Other*. The detailed column is kept.

**How to read the output:** the table shows how many young people fall in each level. Check the *Other* list: any label printed there was not matched by our rules and should be placed by hand if it is large.

In [ ]:
# STEP 25C: GROUP THE DETAILED EDUCATION CODES INTO A FEW LEVELS
import re

EDU_ORDER = [
    "No formal or Quranic/Islamic",
    "Nursery or primary",
    "Junior secondary",
    "Senior secondary or vocational",
    "Polytechnic, NCE or nursing",
    "University or higher",
    "Unknown",
    "Other"
]

def group_education(value):
    """Turn one detailed ed6 label into one of a few education levels."""
    if pd.isna(value):
        return "Unknown"
    v = str(value).strip().upper()
    if v.startswith("UNIVERSITY") or v.startswith("HIGHER DEGREE"):
        return "University or higher"
    if v.startswith(("OND", "HND", "NCE", "NURSING", "POLY")):
        return "Polytechnic, NCE or nursing"
    if re.fullmatch(r"SS[1-3]", v) or v.startswith(("SECONDARY", "TEACHER CERTIFICATE")):
        return "Senior secondary or vocational"
    if re.fullmatch(r"JS[1-3]", v):
        return "Junior secondary"
    if re.fullmatch(r"P[1-6]", v) or re.fullmatch(r"N[1-2]", v) or v.startswith("PRE-NURSERY"):
        return "Nursery or primary"
    if (v == "NONE" or "QURAN" in v or "ISLAM" in v or "TSANGAYA" in v
            or "LITERACY" in v or v.startswith("ADULT")):
        return "No formal or Quranic/Islamic"
    return "Other"

# Add the grouped column to both datasets. The detailed ed6 column is kept.
youth_data["education_level"] = youth_data["ed6"].map(group_education)
eda_data["education_level"] = eda_data["ed6"].map(group_education)

print("Respondents in each education level (all youth):")
display(youth_data["education_level"].value_counts().reindex(EDU_ORDER).dropna().astype(int).to_frame("respondents"))

other_labels = sorted(youth_data.loc[youth_data["education_level"] == "Other", "ed6"].dropna().astype(str).unique())
print("Detailed labels placed in 'Other' (check these by eye):", other_labels if other_labels else "none")


#### **Weighted shares by sector, sex, age and education**

**What this cell does:** prints weighted percentages and ± margins for each group, then draws four stacked bars.

**How to read the output:** each row adds up to 100%. Compare the *Underemployed (%)* column across rows. If two groups differ by less than their ± margins, treat the difference as uncertain.

In [ ]:
# STEP 37W-1: WEIGHTED SHARES WITH 95% CONFIDENCE INTERVALS (RESEARCH QUESTION 1)

COLOURS = ["#E76F51", "#E9C46A", "#2A9D8F"]   # unemployed, underemployed, adequately employed

def normalised_weights(df):
    """Scale popw so that the weights of each quarter add up to 1.
    Every quarter then counts equally and quarters are never added together as one survey."""
    raw = pd.to_numeric(df["popw"], errors="coerce").astype(float)
    raw = raw.where(np.isfinite(raw) & (raw > 0))
    return raw / raw.groupby(df["quarter"]).transform("sum")

def weighted_share_table(df, group_col, order=None):
    """Weighted % in each class for every value of group_col, with a 95% margin of error.
    The margin uses the Kish effective sample size, so it allows for unequal weights
    but NOT for the full NLFS clustering. Treat it as a guide, not an exact interval."""
    d = df.copy()
    d["_w"] = normalised_weights(d)
    d = d.loc[d["_w"].notna()]
    rows = []
    for key, g in d.groupby(group_col, observed=True):
        total = g["_w"].sum()
        n_eff = total ** 2 / (g["_w"] ** 2).sum()
        row = {group_col: key, "Respondents": len(g)}
        for cls in category_order:
            p = g.loc[g["employment_label"] == cls, "_w"].sum() / total
            row[f"{cls} (%)"] = 100 * p
            row[f"{cls} ±95%"] = 196 * np.sqrt(p * (1 - p) / n_eff)
        rows.append(row)
    out = pd.DataFrame(rows).set_index(group_col)
    if order is not None:
        out = out.reindex([o for o in order if o in out.index])
    return out.round(1)

overall_w = weighted_share_table(eda_data.assign(group="All classified youth"), "group")
by_sex_w = weighted_share_table(eda_data, "dc3")
by_age_w = weighted_share_table(eda_data, "age_group", order=age_labels)
by_edu_w = weighted_share_table(eda_data, "education_level", order=EDU_ORDER)
by_sector_w = weighted_share_table(eda_data, "id5_sector")

for title, table in [("ALL CLASSIFIED YOUTH", overall_w), ("BY SEX", by_sex_w),
                     ("BY AGE GROUP", by_age_w), ("BY EDUCATION LEVEL", by_edu_w),
                     ("BY URBAN / RURAL SECTOR", by_sector_w)]:
    print(f"\nWEIGHTED SHARES {title}")
    display(table)

def stacked_panel(ax, table, title):
    pct = table[[f"{c} (%)" for c in category_order]]
    left = np.zeros(len(pct))
    for cls, colour in zip(category_order, COLOURS):
        vals = pct[f"{cls} (%)"].to_numpy(dtype=float)
        ax.barh(range(len(pct)), vals, left=left, color=colour, label=cls)
        for i, (v, l) in enumerate(zip(vals, left)):
            if v >= 6:
                ax.text(l + v / 2, i, f"{v:.0f}%", ha="center", va="center", fontsize=8)
        left = left + vals
    ax.set_yticks(range(len(pct)))
    ax.set_yticklabels([str(i)[:30] for i in pct.index], fontsize=8)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Weighted % of classified youth")
    ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
stacked_panel(axes[0, 0], by_sector_w, "Urban / rural sector")
stacked_panel(axes[0, 1], by_sex_w, "Sex")
stacked_panel(axes[1, 0], by_age_w, "Age group")
stacked_panel(axes[1, 1], by_edu_w, "Education level")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)
fig.suptitle("Survey-weighted employment categories (each quarter weighted equally)", fontsize=12)
plt.tight_layout(rect=[0, 0.05, 1, 0.96])
plt.show()


#### **Weighted shares by state**

**What this cell does:** ranks all states by their weighted underemployment share and charts the ten highest and ten lowest. Only states with at least 30 classified respondents are charted, because very small samples give unstable percentages.

**How to read the output:** a state with a low underemployment share can still have a high unemployment share, so look at both columns. State figures are from the survey sample and are **not** official state rates.

In [ ]:
# STEP 37W-2: WEIGHTED SHARES BY STATE

by_state_w = weighted_share_table(eda_data, "id2_state")
state_ranked = by_state_w.sort_values("Underemployed (%)", ascending=False)
print("All states, sorted by weighted underemployment share:")
display(state_ranked)

# Small samples give unstable percentages, so the chart uses only states with 30+ respondents.
reliable_states = state_ranked.loc[state_ranked["Respondents"] >= 30]
print(f"\nStates with at least 30 classified respondents: {len(reliable_states)} of {len(state_ranked)}")

k = min(10, len(reliable_states) // 2)          # 10 normally; fewer if few states are reliable
top10 = reliable_states.head(k)
bottom10 = reliable_states.tail(k)
chart_states = pd.concat([top10, bottom10])

fig, ax = plt.subplots(figsize=(9, 8))
colours = ["#E76F51"] * len(top10) + ["#2A9D8F"] * len(bottom10)
ax.barh(range(len(chart_states)), chart_states["Underemployed (%)"].to_numpy(dtype=float),
        xerr=chart_states["Underemployed ±95%"].to_numpy(dtype=float), color=colours, capsize=2)
ax.set_yticks(range(len(chart_states)))
ax.set_yticklabels([f"{s} (n={n})" for s, n in zip(chart_states.index, chart_states["Respondents"])], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Weighted underemployed share (%), with 95% margin")
ax.set_title("States with the highest (red) and lowest (green) underemployment share")
plt.tight_layout()
plt.show()


#### **What kind of job do people in each class have? (the informality lens)**

About 90% of workers are not employees, so job type cannot separate the classes by itself. We use it only to **describe** each class. The unemployed class is not shown because those people have no job.

**How to read the output:** each row adds up to 100%. For example, if "Helping in a household business" is a large part of the underemployed row but a small part of the adequately employed row, unpaid or low-paid family work is a main route into underemployment.

In [ ]:
# STEP 37W-3: JOB-STATUS MIX WITHIN EACH CLASS (INFORMALITY LENS)
# mjj4 is the main-job arrangement. Unemployed people have no job, so only the two employed classes appear.

jobs = eda_data.loc[eda_data["mjj4"].notna()].copy()
jobs["_w"] = normalised_weights(jobs)
jobs = jobs.loc[jobs["_w"].notna()]

mix_weights = jobs.groupby(["employment_label", "mjj4"], observed=True)["_w"].sum().unstack(fill_value=0)
mix_percent = (mix_weights.div(mix_weights.sum(axis=1), axis=0) * 100).round(1)
mix_percent.index = mix_percent.index.astype(str)
mix_counts = jobs.groupby(["employment_label", "mjj4"], observed=True).size().unstack(fill_value=0)
mix_counts.index = mix_counts.index.astype(str)

print("Weighted % of each employed class by type of main job (each row adds to 100):")
display(mix_percent)
print("Unweighted respondent counts:")
display(mix_counts)

short = {c: str(c)[:42] for c in mix_percent.columns}
ax = mix_percent.rename(columns=short).plot(kind="barh", stacked=True, figsize=(11, 4), colormap="Set2")
ax.invert_yaxis()
ax.set_xlabel("Weighted % of the class")
ax.set_title("What kind of job do underemployed and adequately employed youth have?")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


#### **Does the low-pay line change the story?**

Our low-pay line (two-thirds of the quarterly median) is a **research choice**, not an official poverty line or minimum wage. Here we move the line from 50% to 100% of the median and rebuild the classes each time with the same rules.

**How to read the output:** the row for 0.667 must match our real results (the cell prints MATCH). If the underemployed share stays high even at the lowest line, the main finding does not depend on this choice.

In [ ]:
# STEP 37W-4: SENSITIVITY OF THE RESULTS TO THE LOW-PAY THRESHOLD
# Our threshold is two-thirds of the quarterly median. Here we try other fractions of the median
# and rebuild the three classes each time, using exactly the same rules as STEPS 11A, 11B and 12A.

quarter_median = youth_data["low_pay_threshold"] / (2 / 3)      # the median we used for each respondent
earnings = youth_data["monthly_earnings_reviewed"]
earnings_known = (youth_data["has_job"] & ~youth_data["unpaid_employment"] & earnings.notna())

fractions = [0.50, 0.60, 2 / 3, 0.75, 1.00]
sens_rows = []

for fraction in fractions:
    low_m = pd.Series(pd.NA, index=youth_data.index, dtype="boolean")
    low_m.loc[earnings_known] = (earnings[earnings_known] < (quarter_median * fraction)[earnings_known])

    confirmed_m = youth_data["has_job"] & (
        youth_data["time_underemployed"] | youth_data["unpaid_employment"] | low_m.fillna(False)
    )
    unresolved_m = youth_data["has_job"] & ~confirmed_m & low_m.isna()
    adequate_m = youth_data["has_job"] & ~confirmed_m & ~unresolved_m

    labelled = youth_data[["quarter", "popw"]].copy()
    labelled["cls"] = np.nan
    labelled.loc[youth_data["unemployed"], "cls"] = 0
    labelled.loc[confirmed_m, "cls"] = 1
    labelled.loc[adequate_m, "cls"] = 2
    labelled = labelled.loc[labelled["cls"].notna()]
    labelled["_w"] = normalised_weights(labelled)
    labelled = labelled.loc[labelled["_w"].notna()]
    share = labelled.groupby("cls")["_w"].sum() / labelled["_w"].sum() * 100

    sens_rows.append({
        "Low-pay line (fraction of median)": round(fraction, 3),
        "Underemployed (count)": int(confirmed_m.sum()),
        "Adequately employed (count)": int(adequate_m.sum()),
        "Unemployed (weighted %)": round(share.get(0, 0), 1),
        "Underemployed (weighted %)": round(share.get(1, 0), 1),
        "Adequately employed (weighted %)": round(share.get(2, 0), 1),
    })

sensitivity_table = pd.DataFrame(sens_rows).set_index("Low-pay line (fraction of median)")
display(sensitivity_table)

# Safety check: the 2/3 row must reproduce the classes already in youth_data.
actual_under = int((youth_data["employment_class"] == 1).sum())
actual_adequate = int((youth_data["employment_class"] == 2).sum())
row_two_thirds = sensitivity_table.loc[round(2 / 3, 3)]
print("Check against the real target at 2/3:",
      "MATCH" if (row_two_thirds["Underemployed (count)"] == actual_under
                  and row_two_thirds["Adequately employed (count)"] == actual_adequate)
      else f"DIFFERENT (actual {actual_under} / {actual_adequate})")

ax = sensitivity_table[["Underemployed (weighted %)", "Adequately employed (weighted %)",
                        "Unemployed (weighted %)"]].plot(kind="line", marker="o", figsize=(8, 4.5),
                                                          color=["#E9C46A", "#2A9D8F", "#E76F51"])
ax.set_ylabel("Weighted % of classified youth")
ax.set_title("How the class shares change with the low-pay line")
plt.tight_layout()
plt.show()


## **5B. Follow-up checks: employment screening and agricultural work**

The checks in this part were run after the exploratory analysis. They look at the 2,039 respondents who answered YES to a work-screening question but have no main-job arrangement (`mjj4`). Understanding them is needed before we can decide how to treat them in the model. The step names (STEP 2 to STEP 5.4) belong to this follow-up investigation.

In [ ]:
# STEP 2: INVESTIGATE EMPLOYMENT CLASSIFICATION QUESTIONS

import pandas as pd

questions = [
    "mjj4",
    "agf2a",
    "sjj5",
    "sjj5b",
    "sjj5c",
    "sjj5d",
    "sjj7",
    "sjj8",
    "sjj10",
    "sjj12",
    "sjj12a",
    "mjj8b_9",
    "um1_1",
    "um1_2",
    "um10a",
    "um10b"
]

for col in questions:

    print("\n" + "=" * 70)
    print("QUESTION:", col)

    if col not in youth_data.columns:
        print("Column not found.")
        continue

    print("\nResponse counts:")

    print(
        youth_data[col]
        .value_counts(dropna=False)
        .head(20)
    )

    print("\nPercentage missing:")

    print(
        round(
            youth_data[col].isna().mean() * 100,
            2
        ),
        "%"
    )

In [ ]:
# STEP 3: CHECK OFFICIAL NBS QUESTION LABELS

!pip -q install pyreadstat

import pyreadstat
import glob
import os

# Locate the Q3 2024 individual dataset
stata_files = glob.glob(
    "/content/NLFS_Dataset/**/nlfs_2024q3_indiv.dta",
    recursive=True
)

if not stata_files:
    raise FileNotFoundError("Q3 2024 dataset not found.")

stata_path = stata_files[0]

# Read the dataset metadata
df_metadata, metadata = pyreadstat.read_dta(
    stata_path,
    metadataonly=True
)

# Display employment-related question descriptions
print("=" * 75)
print("OFFICIAL NBS QUESTION DESCRIPTIONS")
print("=" * 75)

for variable, label in zip(
    metadata.column_names,
    metadata.column_names_to_labels.values()
):

    name = variable.lower()

    if name.startswith(("atw", "abs", "agf", "mjj", "sjj", "um")):

        print(f"{variable}: {label}")

print("\n" + "=" * 75)
print("IMPORTANT VALUE LABELS")
print("=" * 75)

selected = [
    "mjj4", "agf2a", "sjj7",
    "sjj8", "mjj8b_9",
    "um1_1", "um1_2",
    "um10a", "um10b"
]

for variable in selected:

    if variable in metadata.variable_value_labels:

        print(f"\n{variable}")

        print(
            metadata.variable_value_labels[variable]
        )

In [ ]:
# STEP 3: CHECK EMPLOYMENT SCREENING RESPONSES

screening_columns = [
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b",
    "agf1a_1", "agf1a_2", "agf1a_3", "agf1a_4",
    "agf1b_1", "agf1b_2", "agf1b_3", "agf1b_4",
    "agf2d", "abs2", "abs3", "abs4"
]

for col in screening_columns:
    if col in youth_data.columns:
        print(f"\n{col}")
        print(youth_data[col].value_counts(dropna=False).to_string())

### 5B.1 Employment screening and identification

Before classifying respondents into labour market categories, we need to identify individuals who participated in economic activities.

The NLFS questionnaire contains several employment screening questions:

- `atw1`: Work performed for someone else for pay.
- `atw2`: Participation in a business or self-employment activity.
- `atw3`: Assistance in a household business.
- `agf1b_1` to `agf1b_4`: Agricultural employment-related responses.
- `abs1a` and `abs1b`: Information about having a job or unpaid work despite temporary absence.

These variables help identify potential employment that may not be captured by the main job arrangement variable (`mjj4`).

In this stage, we create a preliminary employment indicator to identify respondents with evidence of work. This is a diagnostic step, not the final labour market classification. Survey routing and agricultural activity definitions will be verified before constructing the final target variable.

In [ ]:
# STEP 4: CREATE PRELIMINARY EMPLOYMENT INDICATOR

def is_yes(column):
    return (
        youth_data[column]
        .astype("string")
        .str.strip()
        .str.upper()
        .eq("YES")
        .fillna(False)
    )

# Direct employment screening
youth_data["work_screen_yes"] = (
    is_yes("atw1") |
    is_yes("atw2") |
    is_yes("atw3")
)

# Agricultural activity evidence
agriculture_yes = (
    is_yes("agf1b_1") |
    is_yes("agf1b_2") |
    is_yes("agf1b_3") |
    is_yes("agf1b_4")
)

# Temporary absence evidence
absence_yes = (
    is_yes("abs1a") |
    is_yes("abs1b")
)

# Combine employment evidence
youth_data["preliminary_employed"] = (
    youth_data["work_screen_yes"] |
    agriculture_yes |
    absence_yes
)

# Display results
print("PRELIMINARY EMPLOYMENT RESULTS")

print(
    youth_data["preliminary_employed"]
    .value_counts()
)

print("\nEmployment evidence by quarter:")

display(
    pd.crosstab(
        youth_data["quarter"],
        youth_data["preliminary_employed"]
    )
)

print("\nEmployment evidence with missing main job arrangement:")

print(
    (
        youth_data["preliminary_employed"] &
        youth_data["mjj4"].isna()
    ).sum()
)

### 5B.2 Investigating missing main-job information

The variable `mjj4` describes the arrangement of a respondent's main job. However, our preliminary employment indicator identified 2,039 respondents who have evidence of employment but have missing values for this variable.

This may occur because of questionnaire routing, agricultural activities, or other employment arrangements.

We will investigate these respondents to understand which screening questions identified their employment. This will help prevent incorrect employment classifications and improve the reliability of our target variable.

In [ ]:
# STEP 5: INVESTIGATE EMPLOYMENT EVIDENCE WITH MISSING MJJ4

missing_main_job = youth_data[
    youth_data["preliminary_employed"] &
    youth_data["mjj4"].isna()
].copy()

investigation_columns = [
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b",
    "agf1b_1", "agf1b_2",
    "agf1b_3", "agf1b_4",
    "agf2d", "quarter"
]

print("Records investigated:", len(missing_main_job))

print("\nEmployment screening responses:")

for col in investigation_columns:
    print(f"\n{col}")
    print(
        missing_main_job[col]
        .value_counts(dropna=False)
    )

print("\nEmployment evidence combinations:")

evidence = pd.DataFrame({
    "ATW1": is_yes("atw1"),
    "ATW2": is_yes("atw2"),
    "ATW3": is_yes("atw3"),
    "Agriculture": agriculture_yes,
    "Temporary_absence": absence_yes
}, index=youth_data.index)

display(
    evidence.loc[missing_main_job.index]
    .value_counts()
    .head(15)
    .to_frame("Number of respondents")
)

In [ ]:
# STEP 5.1: INSPECT OFFICIAL NBS QUESTION LABELS
# Skip files that cannot be read.

import os
import glob
import pyreadstat

dta_files = glob.glob(
    "/content/NLFS_Dataset/**/*.dta",
    recursive=True
)

variables_to_check = [
    "atw1", "atw2", "atw3",
    "abs1a", "abs1b", "abs2", "abs3", "abs4",
    "agf1a_1", "agf1a_2", "agf1a_3", "agf1a_4",
    "agf1b_1", "agf1b_2", "agf1b_3", "agf1b_4",
    "agf2a", "agf2d"
]

for file_path in dta_files:

    filename = os.path.basename(file_path)

    print("\n" + "=" * 70)
    print("FILE:", filename)
    print("=" * 70)

    try:
        _, metadata = pyreadstat.read_dta(
            file_path,
            metadataonly=True
        )

    except Exception as error:
        print("SKIPPED: This file could not be read.")
        print("Reason:", error)
        continue

    for variable in variables_to_check:

        if variable not in metadata.column_names:
            continue

        question = metadata.column_names_to_labels.get(
            variable,
            "No question label available"
        )

        label_set_name = metadata.variable_to_label.get(variable)

        print(f"\n{variable}: {question}")

        if label_set_name:
            response_labels = metadata.value_labels.get(
                label_set_name, {}
            )

            print("Response categories:", response_labels)

In [ ]:
# STEP 5.2: AGRICULTURAL ACTIVITY INVESTIGATION

# Recreate records with preliminary employment evidence
# but missing main-job information.

missing_main_job = youth_data.loc[
    youth_data["preliminary_employed"].fillna(False)
    & youth_data["mjj4"].isna()
].copy()

print("Records being investigated:", len(missing_main_job))

# Agricultural activity variables
activity_columns = [
    "agf1a_1",
    "agf1a_2",
    "agf1a_3",
    "agf1a_4"
]

print("\n1. AGRICULTURAL ACTIVITIES")
print("=" * 60)

for col in activity_columns:
    print(f"\n{col}")
    print(missing_main_job[col].value_counts(dropna=False))

# Agricultural work type
work_type_columns = [
    "agf1b_1",
    "agf1b_2",
    "agf1b_3",
    "agf1b_4"
]

print("\n2. TYPE OF AGRICULTURAL WORK")
print("=" * 60)

for col in work_type_columns:
    print(f"\n{col}")
    print(missing_main_job[col].value_counts(dropna=False))

# Purpose of agricultural production
print("\n3. PURPOSE OF AGRICULTURAL PRODUCTION")
print("=" * 60)

print(
    missing_main_job["agf2a"]
    .value_counts(dropna=False)
)

# Whether the person was hired
print("\n4. WHETHER THE PERSON WAS HIRED")
print("=" * 60)

print(
    missing_main_job["agf2d"]
    .value_counts(dropna=False)
)

# Compare agricultural responses across quarters
print("\n5. AGRICULTURAL RESPONSES BY QUARTER")
print("=" * 60)

display(
    missing_main_job.groupby(
        "quarter",
        observed=False
    )[["agf2a", "agf2d"]]
    .agg(lambda x: x.value_counts(dropna=False).to_dict())
)

In [ ]:
# STEP 5.3: CHECK AGRICULTURAL RESPONSE CONSISTENCY

# Agricultural variables to compare
pairs = [
    ("agf1a_1", "agf1b_1", "Crop farming"),
    ("agf1a_2", "agf1b_2", "Rearing farm animals"),
    ("agf1a_3", "agf1b_3", "Fishing or fish farming")
]

# First inspect the full youth dataset
print("AGRICULTURAL RESPONSES IN THE FULL YOUTH DATASET")
print("=" * 70)

for first, second, activity in pairs:

    print(f"\n{activity}: {first} compared with {second}")

    print(pd.crosstab(
        youth_data[first],
        youth_data[second],
        dropna=False
    ))

# Then inspect the 2,039 records separately
print("\n\nAGRICULTURAL RESPONSES IN THE 2,039 INVESTIGATED RECORDS")
print("=" * 70)

for first, second, activity in pairs:

    print(f"\n{activity}: {first} compared with {second}")

    print(pd.crosstab(
        missing_main_job[first],
        missing_main_job[second],
        dropna=False
    ))

# Compare the response counts by quarter
print("\n\nAGRICULTURAL RESPONSES BY QUARTER")
print("=" * 70)

for col in [
    "agf1a_1", "agf1a_2", "agf1a_3", "agf1a_4",
    "agf1b_1", "agf1b_2", "agf1b_3", "agf1b_4"
]:

    print(f"\n{col}")

    display(pd.crosstab(
        youth_data["quarter"],
        youth_data[col],
        dropna=False
    ))

In [ ]:
# STEP 5.4: INVESTIGATE AGRICULTURAL QUESTIONNAIRE ROUTING

print("1. ATW2 (RUNNING A BUSINESS) VS AGRICULTURAL WORK TYPE")
print("=" * 70)

for col in [
    "agf1b_1", "agf1b_2",
    "agf1b_3", "agf1b_4"
]:
    print(f"\nATW2 compared with {col}")

    display(pd.crosstab(
        missing_main_job["atw2"],
        missing_main_job[col],
        dropna=False
    ))


print("\n2. ATW3 (HELPING A HOUSEHOLD BUSINESS) VS AGRICULTURAL WORK TYPE")
print("=" * 70)

for col in [
    "agf1b_1", "agf1b_2",
    "agf1b_3", "agf1b_4"
]:
    print(f"\nATW3 compared with {col}")

    display(pd.crosstab(
        missing_main_job["atw3"],
        missing_main_job[col],
        dropna=False
    ))


print("\n3. BUSINESS ACTIVITY VS AGRICULTURAL PRODUCTION PURPOSE")
print("=" * 70)

for col in ["atw2", "atw3"]:
    print(f"\n{col} compared with agf2a")

    display(pd.crosstab(
        missing_main_job[col],
        missing_main_job["agf2a"],
        dropna=False
    ))


print("\n4. AGRICULTURAL WORK TYPE BY QUARTER")
print("=" * 70)

for col in [
    "agf1b_1", "agf1b_2",
    "agf1b_3", "agf1b_4"
]:
    print(f"\n{col}")

    display(pd.crosstab(
        missing_main_job["quarter"],
        missing_main_job[col],
        dropna=False
    ))

# 6. Model: final data validation and machine learning

This section continues from the questionnaire and identifier investigation above. Run the notebook from the top in Google Colab so that all variables are available.

**Important research distinction:** NBS defines *time-related underemployment* using working fewer than 40 hours per week, wanting additional hours, and being available to work more. Our proposal deliberately uses a broader, project-defined underutilisation class: a worker is placed in that class if they meet the time-shortfall rule, report no payment, or fall below the project earnings threshold. This is **not** the official NBS time-related underemployment rate. We must describe it as our operational/project-defined class in the report and defence. See the NBS definitions and FAQ: https://nigerianstat.gov.ng/page/nlfs-faq and the NBS microdata catalogue: https://microdata.nigerianstat.gov.ng/index.php/catalog/152.

We also found 2,039 records with employment-screening evidence but missing `mjj4`. We will not silently call these people unemployed. For this first modelling run, we exclude those uncertain records from model training and report their number separately. Their final treatment requires questionnaire-routing confirmation.


## 6.1 Confirm that the expected datasets and columns exist

**What this cell does:** checks that the combined youth dataset, quarter, age, target, weight and demographic predictors are present. It stops with a clear error if an earlier cell was skipped or if the data was not rebuilt.


In [ ]:
# Check whether the main dataset exists
print("youth_data exists:", "youth_data" in globals())
print("validation_data exists:", "validation_data" in globals())
print("employment_class exists as a separate variable:", "employment_class" in globals())

# Check the dataset
if "youth_data" in globals():

    print("\nDataset shape:", youth_data.shape)

    print("\nAvailable columns:")
    print(youth_data.columns.tolist())

    print("\nRequired column check:")

    required_columns = [
        "quarter",
        "dc5",
        "dc3",
        "id2_state",
        "id5_sector",
        "popw",
        "employment_class"
    ]

    for col in required_columns:
        print(f"{col}: {'Available' if col in youth_data.columns else 'MISSING'}")

**How to interpret the output:** we expect the three selected quarters (2024q3, 2025q1 and 2025q2), ages 15–34, and a combined dataset close to the previously confirmed 38,792 rows. If the cell raises an error, stop and fix the missing item rather than continuing.


## 6.2 Data quality checks before modelling

**What this cell does:** runs the checks promised in the proposal: age range, impossible working hours (more than 168 a week), workers with no recorded hours, identical rows, and whether category labels match in every quarter.

**How to read the output:** we expect no ages outside 15–34 and no hours above 168. Workers with no recorded hours cannot meet the hours rule, so they can only be underemployed through no pay or low pay. Identical rows are only flagged, never deleted.

In [ ]:
# STEP 37Q: DATA QUALITY CHECKS BEFORE MODELLING
print("1. AGE RANGE")
age_numeric = pd.to_numeric(youth_data["dc5"].astype("string"), errors="coerce")
print("   Missing ages:", int(age_numeric.isna().sum()))
print("   Youngest and oldest:", age_numeric.min(), "and", age_numeric.max())
print("   Records outside 15-34:", int((age_numeric.notna() & ~age_numeric.between(15, 34)).sum()))

print("\n2. IMPOSSIBLE WORKING HOURS")
hours = pd.to_numeric(youth_data["weekly_hours"], errors="coerce")
print("   Records with more than 168 hours in a week (impossible):", int(hours.gt(168).sum()))
print("   Records with more than 100 hours (unusual but possible):", int(hours.gt(100).sum()))

print("\n3. WORKERS WITH NO RECORDED WEEKLY HOURS")
job_no_hours = youth_data["has_job"] & hours.isna()
display(pd.crosstab(youth_data["quarter"], job_no_hours).rename(
    columns={False: "Hours recorded", True: "Has job but no hours"}))
print("   Their class in the target (<NA> means not classified):")
display(youth_data.loc[job_no_hours, "employment_class"].value_counts(dropna=False).to_frame("respondents"))
print("   These people cannot meet the hours rule, so they can only be underemployed through no pay or low pay.")

print("\n4. FULLY IDENTICAL ROWS")
print("   Rows identical to an earlier row:", int(youth_data.duplicated().sum()),
      "(identical rows are not necessarily errors; they are only flagged)")

print("\n5. DO CATEGORY LABELS MATCH ACROSS QUARTERS?")
for col in ["dc3", "id5_sector", "id2_state", "ed6"]:
    labels_by_quarter = {
        q: set(youth_data.loc[youth_data["quarter"] == q, col].dropna().astype(str).unique())
        for q in sorted(youth_data["quarter"].astype(str).unique())
    }
    all_labels = set().union(*labels_by_quarter.values())
    missing_somewhere = {q: sorted(all_labels - s)[:10] for q, s in labels_by_quarter.items() if all_labels - s}
    print(f"   {col}:", "consistent in all quarters" if not missing_somewhere else missing_somewhere)


## 6.3 Rebuild the household identifiers and check the household-roster key

**What the first cell does:** reads the identifier columns again from the original NBS files, applies exactly the same sample and age filters as STEP 1, and checks that the rows line up one-to-one with `youth_data` (row count, quarter, age and sample label). If any check fails, the cell stops.

**What the second cell does:** prints the number of interview IDs linked to more than one household combination, the rows in repeated household-roster combinations, and how many household codes appear in more than one quarter. A repeated roster key is an investigation flag, not proof that a person is duplicated. We keep the records unless the survey documentation shows a true duplicate.

In [ ]:

# STEP 37A-2: REBUILD AND ALIGN HOUSEHOLD IDENTIFIERS

import pandas as pd
import numpy as np

main_sample_labels = {
    "2024q3": "NLFS target",
    "2025q1": "Original",
    "2025q2": "Original"
}

id_columns = [
    "interview_id",
    "hhroster_id",
    "id4b_ea",
    "id7a_structure",
    "id7_hhnumber",
    "dc5",
    "sample"
]

parts = []

# Use the same quarter order as the original youth_data.
expected_quarters = ["2024q3", "2025q1", "2025q2"]

assert list(FILES.keys()) == expected_quarters, (
    f"Unexpected FILES order: {list(FILES.keys())}"
)

for quarter in expected_quarters:

    file_path = FILES[quarter]

    part = pd.read_stata(
        file_path,
        columns=id_columns,
        convert_categoricals=True
    )

    age = pd.to_numeric(
        part["dc5"].astype("string"),
        errors="coerce"
    )

    sample_values = (
        part["sample"].astype("string").str.strip()
    )

    selected = (
        age.between(15, 34)
        & sample_values.eq(main_sample_labels[quarter])
    )

    part = part.loc[selected].copy()
    part["quarter"] = quarter

    parts.append(part)

    print(f"{quarter}: {len(part):,} records selected")

# Combine the identifier records in the original quarter order.
identifier_data = pd.concat(
    parts,
    ignore_index=True
)

# Confirm row counts.
assert len(identifier_data) == len(youth_data), (
    f"Row count mismatch: identifier_data={len(identifier_data)}, "
    f"youth_data={len(youth_data)}"
)

# Compare quarter order.
assert np.array_equal(
    identifier_data["quarter"].astype(str).to_numpy(),
    youth_data["quarter"].astype(str).to_numpy()
), "Quarter alignment failed."

# Compare ages numerically to avoid data-type comparison errors.
identifier_age = pd.to_numeric(
    identifier_data["dc5"].astype("string"),
    errors="coerce"
).to_numpy()

youth_age = pd.to_numeric(
    youth_data["dc5"].astype("string"),
    errors="coerce"
).to_numpy()

assert np.array_equal(
    identifier_age,
    youth_age,
    equal_nan=True
), "Age alignment failed."

# Compare sample labels.
assert np.array_equal(
    identifier_data["sample"].astype("string").str.strip().to_numpy(),
    youth_data["sample"].astype("string").str.strip().to_numpy()
), "Sample alignment failed."

# Assign the original youth_data index.
identifier_data.index = youth_data.index

# Final alignment checks.
assert identifier_data.index.equals(youth_data.index)
assert identifier_data.index.is_unique
assert youth_data.index.is_unique

print("\nSTEP 37A-2 COMPLETED.")
print("Youth dataset:", youth_data.shape)
print("Identifier dataset:", identifier_data.shape)
print("Index alignment: confirmed")




In [ ]:
# STEP 37B: HOUSEHOLD-ROSTER IDENTIFIER DIAGNOSTIC
household_key_columns = ["id4b_ea", "id7a_structure", "id7_hhnumber"]
roster_key_columns = ["quarter"] + household_key_columns + ["hhroster_id"]
needed = ["interview_id"] + household_key_columns + ["hhroster_id", "quarter"]
missing_identifier_columns = [c for c in needed if c not in identifier_data.columns]
assert not missing_identifier_columns, f"Missing identifier columns: {missing_identifier_columns}"

print("Missing identifier values:")
display(identifier_data[needed].isna().sum().to_frame("missing"))

repeated_roster_mask = identifier_data.duplicated(subset=roster_key_columns, keep=False)
print("Rows involved in repeated household-roster combinations:", int(repeated_roster_mask.sum()))

household_key = (
    identifier_data["id4b_ea"].astype(str) + "_"
    + identifier_data["id7a_structure"].astype(str) + "_"
    + identifier_data["id7_hhnumber"].astype(str)
)

households_per_interview = household_key.groupby(
    [identifier_data["quarter"], identifier_data["interview_id"]], dropna=False
).nunique()
print("Interview IDs linked to more than one household combination:", int((households_per_interview > 1).sum()))

household_quarters = pd.DataFrame({
    "key": household_key.to_numpy(),
    "quarter": identifier_data["quarter"].to_numpy()
})
number_of_quarters = household_quarters.groupby("key")["quarter"].nunique()
print("\nHousehold codes in total:", len(number_of_quarters))
print("Household codes that appear in 2 or more quarters:", int((number_of_quarters >= 2).sum()))
print("Household codes that appear in all 3 quarters:", int((number_of_quarters == 3).sum()))
print("No records have been deleted.")


**Interpretation:** the earlier run found 1,145 rows involved in repeated household-roster combinations and no missing identifier values. Different interview IDs sharing a roster key need investigation; they are not automatically duplicate individuals. Do not use these IDs as model predictors.

If the same household code appears in more than one quarter, those rows may describe the same household. For this reason the household groups used for splitting (section 6.7) ignore the quarter, so a household can never be on both sides of the split.

## 6.4 Reconcile the employment-screening discrepancy

**What this cell does:** identifies respondents who answered YES to at least one direct work/temporary-absence screening question but have no `mjj4` main-job arrangement. Earlier diagnostics found 2,039 such records. Because `mjj4` alone is not enough to resolve these questionnaire paths, the cell marks them as uncertain and excludes them from the first model sample.


In [ ]:
# STEP 37C: FLAG EMPLOYMENT EVIDENCE WITH MISSING MAIN-JOB ARRANGEMENT

# Convert YES/NO-style responses safely to Boolean indicators.
def response_is_yes(series):
    return series.astype("string").str.strip().str.upper().eq("YES").fillna(False)

# Use only screening columns that exist in the combined dataset.
screening_columns = [c for c in ["atw1", "atw2", "atw3", "abs1a", "abs1b"] if c in youth_data.columns]
assert screening_columns, "No employment screening columns found. Rebuild youth_data with the screening variables."

# Any YES is evidence that needs consideration; it is not by itself a final employment decision.
direct_screening_evidence = pd.Series(False, index=youth_data.index)
for column in screening_columns:
    direct_screening_evidence |= response_is_yes(youth_data[column])

# mjj4 is the main-job arrangement variable used by the earlier notebook.
has_recorded_main_job = youth_data["mjj4"].notna()
# Prefer the broader preliminary flag already created earlier, because it
# also includes agricultural work evidence. If it is unavailable, use direct
# screening evidence only and print a warning.
if "preliminary_employed" in youth_data.columns:
    any_employment_evidence = youth_data["preliminary_employed"].fillna(False).astype(bool)
else:
    any_employment_evidence = direct_screening_evidence
    print("WARNING: preliminary_employed is unavailable; agricultural evidence may not be included.")

uncertain_employment = any_employment_evidence & ~has_recorded_main_job

print("Direct screening evidence but missing mjj4:", int(uncertain_employment.sum()))
print("By quarter:")
display(youth_data.loc[uncertain_employment].groupby("quarter").size().to_frame("records"))

# Keep the flag for transparent exclusion from the first ML run.
youth_data["employment_screening_unresolved"] = uncertain_employment


**Interpretation:** if the count is still 2,039, the earlier investigation is reproduced. These respondents remain in the source dataset and descriptive audit, but are excluded from model fitting until the employment routing is settled. This is safer than teaching the model incorrect labels.


## 6.5 Audit the three target classes

**What this cell does:** checks the current target distribution and confirms that the labels match the project proposal. The target codes are 0 = unemployed, 1 = project-defined underemployed/underutilised, and 2 = project-defined adequately employed. The class 1 definition combines time-related hours shortfall, no payment, or low earnings.


In [ ]:
# STEP 37D: AUDIT THE THREE-CLASS TARGET

class_names = {0: "Unemployed", 1: "Project-defined underemployed", 2: "Project-defined adequately employed"}

# Convert target to numeric nullable integer without changing its meaning.
youth_data["employment_class"] = pd.to_numeric(
    youth_data["employment_class"], errors="coerce"
).astype("Int64")

print("Target counts (unweighted):")
display(youth_data["employment_class"].map(class_names).fillna("Not classified").value_counts().to_frame("respondents"))

print("Target counts by quarter:")
display(pd.crosstab(
    youth_data["quarter"],
    youth_data["employment_class"].map(class_names).fillna("Not classified")
))

# Check that only the three expected numeric classes are present.
observed_classes = set(youth_data["employment_class"].dropna().astype(int).unique())
assert observed_classes.issubset({0, 1, 2}), f"Unexpected target values: {observed_classes}"

print("Observed class codes:", sorted(observed_classes))


**Interpretation:** check that all three classes appear and that none is unexpectedly tiny or absent. A class imbalance is common in classification; we will use macro-F1 and class weighting rather than judging models by accuracy alone. The counts are sample counts, not population estimates.


## 6.6 Create a modelling sample without unresolved employment records

**What this cell does:** keeps only records with a valid three-class target and removes the unresolved screening cases identified above. It also removes rows with invalid or non-positive population weights. The original `youth_data` remains unchanged except for diagnostic columns.


In [ ]:
# STEP 37E: BUILD THE FIRST MODEL-READY SAMPLE

# Keep records with one of the three valid target classes.
model_data = youth_data.loc[youth_data["employment_class"].notna()].copy()

# Exclude respondents whose screening evidence conflicts with missing mjj4.
model_data = model_data.loc[~model_data["employment_screening_unresolved"]].copy()

# Retain valid positive population weights for weighted summaries and optional model fitting.
model_data["popw"] = pd.to_numeric(model_data["popw"], errors="coerce")
model_data = model_data.loc[model_data["popw"].gt(0)].copy()
model_data["employment_class"] = model_data["employment_class"].astype(int)

print("Model sample shape:", model_data.shape)
print("Class distribution:")
display(model_data["employment_class"].map(class_names).value_counts().to_frame("respondents"))
print("Excluded unresolved screening records:", int(uncertain_employment.sum()))
print("Missing target labels excluded:", int(youth_data["employment_class"].isna().sum()))


**Interpretation:** this is the first defensible modelling sample under the current operational rules. It is not the entire youth population: people outside the labour force, people whose class could not be established, and unresolved screening cases are not assigned one of the three target labels. Report these exclusions transparently.


## 6.7 Select predictors and prevent target leakage

**What this cell does:** uses demographic and geographic characteristics that are known independently of the employment outcome. It deliberately excludes job status, hours, earnings, search and availability answers because those variables were used to construct the target. Including them would allow the model to “cheat” by seeing the answer in its input data.


In [ ]:
# STEP 37F: SELECT SAFE PREDICTORS AND PREPARE THE MODEL DATA
import numpy as np
import pandas as pd

assert "model_data" in globals(), "Run STEP 37E first: it creates model_data."
assert "identifier_data" in globals(), "Run the identifier cell in section 6.3 first."
assert "education_level" in model_data.columns, "Run the education-grouping cell in section 5 first."

# 1. Predictors: background facts known independently of the job outcome.
candidate_features = ["dc5", "dc3", "education_level", "id2_state", "id5_sector", "quarter"]
feature_columns = [c for c in candidate_features if c in model_data.columns]
assert "dc5" in feature_columns, "Age (dc5) is required as a predictor."
assert len(feature_columns) >= 4, f"Too few predictors available: {feature_columns}"

# 2. These columns define the target or identify people, so they must never be predictors.
forbidden = {
    "mjj4", "sjj3", "sjj4", "sjj5", "sjj5b", "sjj5c", "sjj5d", "sjj6", "sjj7", "sjj8", "sjj9",
    "sjj10", "sjj12", "sjj12a", "um1_1", "um1_2", "um10a", "um10b", "has_job", "searching",
    "available", "unemployed", "weekly_hours", "time_underemployed", "unpaid_employment",
    "monthly_earnings", "monthly_earnings_reviewed", "low_paid", "confirmed_underemployed",
    "employment_class", "employment_label", "popw", "interview_id", "hhroster_id",
    "id4b_ea", "id7a_structure", "id7_hhnumber"
}
leaked = sorted(set(feature_columns) & forbidden)
assert not leaked, f"Target leakage: these predictors are not allowed: {leaked}"

# 3. Predictors and target.
X = model_data[feature_columns].copy()
y = model_data["employment_class"].astype(int).copy()

# Age must be a plain number. Text predictors become ordinary objects with plain NaN,
# which scikit-learn's imputers handle safely.
X["dc5"] = pd.to_numeric(X["dc5"].astype("string"), errors="coerce").astype(float)
for col in X.columns.drop("dc5"):
    X[col] = X[col].astype("object")
    X[col] = X[col].where(X[col].notna(), np.nan)

# 4. Household groups. They are used ONLY to keep members of one household on the same side of the split.
# The quarter is left out of the key on purpose: if the same household code appears in more
# than one quarter, all of its rows stay together, which is the safer choice against leakage.
id_cols = ["id4b_ea", "id7a_structure", "id7_hhnumber"]
assert model_data.index.isin(identifier_data.index).all(), "Identifier index does not match model_data."
ids = identifier_data.loc[model_data.index, id_cols]
assert ids.notna().all().all(), "Missing household identifiers."
groups = (ids["id4b_ea"].astype(str) + "_" + ids["id7a_structure"].astype(str)
          + "_" + ids["id7_hhnumber"].astype(str))

# 5. Survey weights.
weights = pd.to_numeric(model_data["popw"], errors="coerce").astype(float)
assert weights.notna().all(), "Missing or invalid survey weights."
assert (weights > 0).all(), "Zero or negative survey weights."

assert len(X) == len(y) == len(groups) == len(weights)

print("Predictors selected:", feature_columns)
print("X shape:", X.shape, "| y shape:", y.shape)
print("Target classes:", sorted(y.unique().tolist()))
print("Household groups:", groups.nunique())
print("Missing values per predictor:")
display(X.isna().sum().to_frame("missing"))
print("\nSTEP 37F COMPLETED.")


**Interpretation:** the model inputs are age, sex, **grouped** education level, state, urban or rural sector and survey quarter. Household and interview identifiers are used only to keep related records together when splitting; they are never model features. The cell stops with an error if any forbidden column (anything used to build the target) appears among the predictors.

Marital status, disability indicators and household structure are on the proposal's candidate list, but they were not loaded in STEP 1, so they are **not** used in this version. This is stated as a limitation and a next step.

## 6.8 Split the data by household, not by individual row

**What this cell does:** creates a training and test set while keeping all members of the same household in one split. This reduces the risk that the model sees one household member during training and another member from the same household during testing.


In [ ]:
# ============================================================
# STEP 37G: GROUP-AWARE TRAIN / TEST SPLIT
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

print("STEP 37G: GROUP-AWARE TRAIN / TEST SPLIT")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check that Step 37F created everything we need
# ------------------------------------------------------------

required_objects = ["X", "y", "groups", "weights"]

missing_objects = [
    obj for obj in required_objects
    if obj not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Step 37G cannot run because these objects are missing: "
        + ", ".join(missing_objects)
        + ".\n\n"
        "Please run the corrected Step 37F cell first."
    )

# ------------------------------------------------------------
# 2. Basic consistency checks
# ------------------------------------------------------------

if not (len(X) == len(y) == len(groups) == len(weights)):
    raise ValueError(
        f"Length mismatch detected:\n"
        f"X = {len(X)}\n"
        f"y = {len(y)}\n"
        f"groups = {len(groups)}\n"
        f"weights = {len(weights)}"
    )

if len(X) == 0:
    raise ValueError("X is empty. Step 37F did not produce usable modelling data.")

# Make sure indices are aligned
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)
groups = groups.reset_index(drop=True)
weights = weights.reset_index(drop=True)

# ------------------------------------------------------------
# 3. Create a group-aware train/test split
# ------------------------------------------------------------
#
# GroupShuffleSplit keeps the same household group from appearing
# in both training and testing data.
#
# This helps prevent data leakage between train and test sets.
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

# ------------------------------------------------------------
# 4. Create train/test datasets
# ------------------------------------------------------------

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

w_train = weights.iloc[train_idx].copy()
w_test = weights.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()

# ------------------------------------------------------------
# 5. Check that no group appears in both datasets
# ------------------------------------------------------------

train_groups = set(groups_train)
test_groups = set(groups_test)

overlapping_groups = train_groups.intersection(test_groups)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

print("Training groups:", len(train_groups))
print("Testing groups:", len(test_groups))

print("Overlapping groups:", len(overlapping_groups))

assert len(overlapping_groups) == 0, (
    "ERROR: Some household groups appear in both "
    "training and testing datasets."
)

# ------------------------------------------------------------
# 6. Check class labels
# ------------------------------------------------------------

allowed_classes = {0, 1, 2}

if not set(y_train.unique()).issubset(allowed_classes):
    raise ValueError(
        f"Unexpected class found in training data: "
        f"{sorted(y_train.unique())}"
    )

if not set(y_test.unique()).issubset(allowed_classes):
    raise ValueError(
        f"Unexpected class found in testing data: "
        f"{sorted(y_test.unique())}"
    )

# ------------------------------------------------------------
# 7. Display class distribution
# ------------------------------------------------------------

print("\nTraining classes:")
display(
    y_train.value_counts()
    .sort_index()
    .rename(index=class_names)
    .to_frame("records")
)

print("\nTesting classes:")
display(
    y_test.value_counts()
    .sort_index()
    .rename(index=class_names)
    .to_frame("records")
)

# ------------------------------------------------------------
# 8. Final confirmation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("STEP 37G COMPLETED SUCCESSFULLY")
print("=" * 60)

print(f"Training records: {len(X_train):,}")
print(f"Testing records:  {len(X_test):,}")
print(f"Training groups:  {len(train_groups):,}")
print(f"Testing groups:   {len(test_groups):,}")
print("Household overlap: 0")
print("Group-aware split: PASSED")

**Interpretation:** the test set is data the models do not see while learning. The class counts should be checked before training; if a class is missing from the test set, the evaluation is not reliable and the split strategy must be adjusted.


## 6.9 Build preprocessing and baseline models

**What this cell does:** fills missing numeric/categorical predictor values, scales numeric inputs for logistic regression, converts categories into indicator columns, and defines four models: a majority-class baseline, multinomial logistic regression, decision tree and random forest. The baseline is important: a complex model should beat a simple “always predict the largest class” rule.


In [ ]:
# STEP 37H: PREPARE THE MACHINE-LEARNING PIPELINE

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone

# Identify numeric and categorical columns from the selected predictors.
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = [c for c in feature_columns if c not in numeric_features]

# Numeric preprocessing: fill missing values with median, then standardize.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing: fill missing values and one-hot encode categories.
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Multinomial logistic regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "Decision tree": DecisionTreeClassifier(
        max_depth=8, min_samples_leaf=30, class_weight="balanced", random_state=42
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=250, max_depth=14, min_samples_leaf=10,
        class_weight="balanced_subsample", n_jobs=-1, random_state=42
    )
}

print("Numeric predictors:", numeric_features)
print("Categorical predictors:", categorical_features)
print("Models prepared:", list(models.keys()))


**Interpretation:** preprocessing is fitted only on the training data when each pipeline is trained. This prevents information from the test set leaking into the learning process. Class weights make the models pay more attention to smaller classes.


## 6.10 Train and evaluate the models

**What this cell does:** trains each model, predicts the untouched test set, and compares accuracy, balanced accuracy and macro-F1. Macro-F1 gives equal importance to each class, so it is more informative than accuracy when class sizes differ. We also pass the survey population weights to models that support weighted fitting and report weighted and unweighted evaluation separately where possible.


In [ ]:
# STEP 37I: FIT MODELS AND COMPARE PERFORMANCE

from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             classification_report, confusion_matrix, recall_score)

trained_models = {}
results = []
predictions = {}

# Population weights are huge numbers. Dividing by their average keeps the same relative
# importance but puts them on an ordinary scale (average = 1) for model fitting.
w_train_scaled = w_train / w_train.mean()

for model_name, estimator in models.items():
    # clone(preprocessor) gives every model its OWN preprocessing object, so refitting one
    # model later can never change the preprocessing inside another saved model.
    pipe = Pipeline(steps=[("preprocess", clone(preprocessor)), ("model", estimator)])

    if model_name == "Majority baseline":
        pipe.fit(X_train, y_train)
    else:
        pipe.fit(X_train, y_train, model__sample_weight=w_train_scaled)

    y_pred = pipe.predict(X_test)
    trained_models[model_name] = pipe
    predictions[model_name] = y_pred

    recalls = recall_score(y_test, y_pred, labels=[0, 1, 2], average=None, zero_division=0)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_test, y_pred),
        "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Weighted macro F1": f1_score(
            y_test, y_pred, average="macro", sample_weight=w_test, zero_division=0
        ),
        "Recall: Unemployed": recalls[0],
        "Recall: Underemployed": recalls[1],
        "Recall: Adequately employed": recalls[2],
    })

results_df = pd.DataFrame(results).sort_values("Macro F1", ascending=False)
display(results_df.reset_index(drop=True).round(3))


**Interpretation:** select the model primarily using macro-F1 and class-specific recall, not accuracy alone. If the models perform poorly on the unemployed class, for example, the overall accuracy can still look good when most respondents belong to the employed classes. The weighted macro-F1 reflects the supplied survey population weights; it does not by itself make the sample design fully modelled.


## 6.11 Inspect class-level performance and confusion matrices

**What this cell does:** prints precision, recall and F1 for each class and displays confusion matrices. Rows represent the true class and columns represent the model's prediction. This tells us which groups are being confused.


In [ ]:
# STEP 37J: DETAILED MODEL EVALUATION

import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
best_predictions = predictions[best_model_name]

print("Best model by unweighted macro-F1:", best_model_name)
print("\nClassification report:")
print(classification_report(
    y_test, best_predictions, labels=[0, 1, 2],
    target_names=[class_names[i] for i in [0, 1, 2]], zero_division=0
))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions, labels=[0, 1, 2],
    display_labels=[class_names[i] for i in [0, 1, 2]],
    cmap="Blues", xticks_rotation=25, ax=ax
)
ax.set_title(f"Confusion matrix — {best_model_name}")
plt.tight_layout()
plt.show()


**Interpretation:** focus on recall for each class. Low recall for unemployed youth means many genuinely unemployed respondents are being predicted as employed. Low precision means many respondents predicted to belong to a class actually belong to another class. These errors matter for policy recommendations.


## 6.12 Robustness check: weighted versus unweighted model fits

**What this cell does:** fits logistic regression and random forest twice, once using the survey weights and once without, and compares the same test scores.

**How to read the output:** if the scores are close, the conclusions do not depend on weighting. Large gaps mean the weighting choice matters and must be reported.

In [ ]:
# STEP 37I-2: ROBUSTNESS CHECK — WEIGHTED VERSUS UNWEIGHTED MODEL FITS

robust_rows = []
for name in ["Multinomial logistic regression", "Random forest"]:
    for use_weights in (True, False):
        pipe = Pipeline(steps=[("preprocess", clone(preprocessor)), ("model", clone(models[name]))])
        if use_weights:
            pipe.fit(X_train, y_train, model__sample_weight=w_train_scaled)
        else:
            pipe.fit(X_train, y_train)
        pred = pipe.predict(X_test)
        robust_rows.append({
            "Model": name,
            "Fitted with survey weights": use_weights,
            "Macro F1": f1_score(y_test, pred, average="macro", zero_division=0),
            "Balanced accuracy": balanced_accuracy_score(y_test, pred),
            "Weighted macro F1": f1_score(y_test, pred, average="macro",
                                          sample_weight=w_test, zero_division=0),
        })

robustness_fit_df = pd.DataFrame(robust_rows)
display(robustness_fit_df.round(3))


## 6.13 Examine which predictors matter

**What this cell does:** extracts feature importance from the random forest, if it was successfully trained. Importance shows which input features the model relied on most; it does not prove that a feature causes unemployment or underutilisation.


In [ ]:
# STEP 37K: RANDOM-FOREST FEATURE IMPORTANCE

rf_pipeline = trained_models.get("Random forest")
if rf_pipeline is not None:
    fitted_preprocessor = rf_pipeline.named_steps["preprocess"]
    fitted_rf = rf_pipeline.named_steps["model"]
    transformed_names = fitted_preprocessor.get_feature_names_out()

    importance_df = pd.DataFrame({
        "Feature": transformed_names,
        "Importance": fitted_rf.feature_importances_
    }).sort_values("Importance", ascending=False)

    display(importance_df.head(20))

    top_features = importance_df.head(15).sort_values("Importance")
    plt.figure(figsize=(9, 6))
    plt.barh(top_features["Feature"], top_features["Importance"])
    plt.title("Top random-forest feature importances")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()
else:
    print("Random forest was not available in the trained models.")


**Interpretation:** importance is predictive, not causal. For example, if education is important, we can say education helps distinguish the classes in this dataset; we cannot conclude from this model alone that changing education would cause a particular labour outcome.


## 6.14 Permutation importance on the test set

**What this cell does:** takes the best non-baseline model, shuffles one predictor at a time in the test data, and measures how much the macro-F1 score falls. A big fall means the model relied on that predictor.

**How to read the output:** this is cleaner than the random-forest importance above because it is measured on data the model has not seen, and it gives one number per original predictor (for example, all states together). It shows what helps **prediction**, not what **causes** underemployment.

In [ ]:
# STEP 37K-2: PERMUTATION IMPORTANCE ON THE TEST SET
from sklearn.inspection import permutation_importance

candidates = results_df.loc[results_df["Model"] != "Majority baseline"]
perm_model_name = candidates.iloc[0]["Model"]
perm_model = trained_models[perm_model_name]

perm = permutation_importance(
    perm_model, X_test, y_test,
    scoring="f1_macro", n_repeats=5, random_state=42, n_jobs=-1
)

permutation_df = pd.DataFrame({
    "Predictor": X_test.columns,
    "Drop in macro-F1 when shuffled": perm.importances_mean,
    "Variation (std)": perm.importances_std
}).sort_values("Drop in macro-F1 when shuffled", ascending=False)

print("Model used:", perm_model_name)
display(permutation_df.round(4).reset_index(drop=True))

plot_df = permutation_df.sort_values("Drop in macro-F1 when shuffled")
plt.figure(figsize=(8, 4))
plt.barh(plot_df["Predictor"], plot_df["Drop in macro-F1 when shuffled"],
         xerr=plot_df["Variation (std)"], color="#2A9D8F")
plt.xlabel("Drop in macro-F1 when this predictor is shuffled")
plt.title("Which background factors does the model rely on most?")
plt.tight_layout()
plt.show()


## 6.15 Odds ratios: what is linked to underemployment among employed youth?

**What this cell does:** among employed youth only, compares underemployed (1) with adequately employed (0) using logistic regression. State, sector and survey round are included so that, for example, the effect of sex is compared between people in the same state.

**How to read the output:**
- An **odds ratio above 1** means higher odds of being underemployed than the reference group; **below 1** means lower odds.
- The 95% CI is the range of plausible values. If it contains 1, the link is not clearly different from zero.
- The weighted column is a rougher check using the survey weights. Compare it with the main column; the two should tell the same story.
- The reference groups are printed. Every odds ratio is a comparison against them.

These are **associations**, not causes.

In [ ]:
# STEP 37R: ODDS RATIOS FOR UNDEREMPLOYED (1) VERSUS ADEQUATELY EMPLOYED (0), EMPLOYED YOUTH ONLY

emp = model_data.loc[model_data["employment_class"].isin([1, 2])].copy()
y_bin = (emp["employment_class"] == 1).astype(float).to_numpy()
w_raw = pd.to_numeric(emp["popw"], errors="coerce").astype(float).to_numpy()
w_scaled = w_raw / w_raw.mean()

def most_common(values):
    return values.value_counts().index[0]

reference_levels = {}

def dummies_for(column, label, preferred_reference=None):
    """One 0/1 column per category, leaving out a reference category."""
    values = emp[column].astype("object").where(emp[column].notna(), "Unknown").astype(str)
    reference = preferred_reference if preferred_reference in set(values) else most_common(values)
    block = pd.get_dummies(values, prefix=label, prefix_sep=": ").astype(float)
    block = block.drop(columns=[f"{label}: {reference}"])
    reference_levels[label] = reference
    return block

age_years = pd.to_numeric(emp["dc5"].astype("string"), errors="coerce")
age_years = age_years.fillna(age_years.median())

design = pd.concat([
    (age_years / 5).rename("Age (per 5 years older)"),
    dummies_for("dc3", "Sex"),
    dummies_for("education_level", "Education", preferred_reference="Senior secondary or vocational"),
    dummies_for("id5_sector", "Sector"),
    dummies_for("quarter", "Survey round"),
    dummies_for("id2_state", "State"),
], axis=1).astype(float)

X_design = np.column_stack([np.ones(len(design)), design.to_numpy()])
term_names = ["Intercept"] + design.columns.tolist()

def fit_logit(X, y, weights=None, max_iter=100, ridge=1e-6):
    """Logistic regression by iteratively reweighted least squares.
    Returns coefficients and their standard errors."""
    n, k = X.shape
    w = np.ones(n) if weights is None else weights
    beta = np.zeros(k)
    for _ in range(max_iter):
        eta = X @ beta
        p = 1 / (1 + np.exp(-np.clip(eta, -30, 30)))
        var = np.clip(p * (1 - p), 1e-9, None)
        z = eta + (y - p) / var
        XtW = X.T * (w * var)
        A = XtW @ X + ridge * np.eye(k)
        new_beta = np.linalg.solve(A, XtW @ z)
        if np.max(np.abs(new_beta - beta)) < 1e-8:
            beta = new_beta
            break
        beta = new_beta
    cov = np.linalg.inv(A)
    return beta, np.sqrt(np.diag(cov))

beta, se = fit_logit(X_design, y_bin)
beta_w, _ = fit_logit(X_design, y_bin, weights=w_scaled)

odds_table = pd.DataFrame({
    "Odds ratio": np.exp(beta),
    "95% CI low": np.exp(beta - 1.96 * se),
    "95% CI high": np.exp(beta + 1.96 * se),
    "Weighted odds ratio (approx.)": np.exp(beta_w),
}, index=term_names).drop(index="Intercept")

odds_table["Clearly different from 1?"] = np.where(
    (odds_table["95% CI low"] > 1) | (odds_table["95% CI high"] < 1), "Yes", "No")

shown = odds_table.loc[~odds_table.index.str.startswith("State: ")]
print("Employed youth in this model:", len(emp))
print("Reference groups (odds ratios compare against these):", reference_levels)
print(f"State is included as {int(odds_table.index.str.startswith('State: ').sum())} control terms "
      "(not listed), so the other odds ratios compare people living in the same state.")
display(shown.round(3))

forest = shown.loc[~shown.index.str.startswith("Survey round")].sort_values("Odds ratio")
plt.figure(figsize=(9, 5))
plt.errorbar(forest["Odds ratio"], range(len(forest)),
             xerr=[forest["Odds ratio"] - forest["95% CI low"], forest["95% CI high"] - forest["Odds ratio"]],
             fmt="o", color="#264653", ecolor="#8AB6AD", capsize=3)
plt.axvline(1, color="red", linestyle="--")
plt.yticks(range(len(forest)), forest.index, fontsize=8)
plt.xscale("log")
plt.xlabel("Odds ratio (log scale). Right of the red line = more likely to be underemployed")
plt.title("Factors associated with being underemployed rather than adequately employed")
plt.tight_layout()
plt.show()


**Interpretation:** pick out the odds ratios whose confidence interval does not contain 1 (the table says "Yes"). Describe them as factors *associated with* underemployment. Odds ratios are not the same as percentage differences: an odds ratio of 1.5 does not mean 50% more people.

## 6.16 Temporal validation using Q2 2025 as a later-quarter test

**What this cell does:** trains on Q3 2024 and Q1 2025, then tests on Q2 2025. This is a useful second check because it asks whether patterns learned from earlier quarters generalise to a later quarter. It is not a forecast into the future beyond the observed survey periods.


In [ ]:
# STEP 37L: TEMPORAL HOLDOUT VALIDATION

if "quarter" in model_data.columns and set(model_data["quarter"].astype(str).unique()) >= {"2024q3", "2025q1", "2025q2"}:
    temporal_train_mask = model_data["quarter"].astype(str).isin(["2024q3", "2025q1"])
    temporal_test_mask = model_data["quarter"].astype(str).eq("2025q2")

    assert len(X) == len(model_data), "X and model_data must have the same rows in the same order."
    X_early = X.loc[temporal_train_mask.to_numpy()]
    X_later = X.loc[temporal_test_mask.to_numpy()]
    y_early = model_data.loc[temporal_train_mask, "employment_class"].astype(int)
    y_later = model_data.loc[temporal_test_mask, "employment_class"].astype(int)
    w_early = model_data.loc[temporal_train_mask, "popw"].astype(float)
    w_later = model_data.loc[temporal_test_mask, "popw"].astype(float)
    w_early_scaled = (w_early / w_early.mean()).to_numpy()

    temporal_model = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", RandomForestClassifier(
            n_estimators=250, max_depth=14, min_samples_leaf=10,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42
        ))
    ])
    temporal_model.fit(X_early, y_early, model__sample_weight=w_early_scaled)
    temporal_predictions = temporal_model.predict(X_later)

    baseline_model = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", DummyClassifier(strategy="most_frequent"))
    ])
    baseline_model.fit(X_early, y_early)
    baseline_predictions = baseline_model.predict(X_later)

    temporal_macro_f1 = f1_score(y_later, temporal_predictions, average="macro", zero_division=0)
    temporal_baseline_f1 = f1_score(y_later, baseline_predictions, average="macro", zero_division=0)

    print("Training quarters: 2024q3 and 2025q1")
    print("Holdout quarter: 2025q2")
    print("Temporal holdout records:", len(y_later))
    print("Random forest macro-F1:", round(temporal_macro_f1, 4))
    print("Majority baseline macro-F1:", round(temporal_baseline_f1, 4))
    print("Random forest weighted macro-F1:", round(f1_score(
        y_later, temporal_predictions, average="macro", sample_weight=w_later, zero_division=0
    ), 4))
    print(classification_report(
        y_later, temporal_predictions, labels=[0, 1, 2],
        target_names=[class_names[i] for i in [0, 1, 2]], zero_division=0
    ))
else:
    print("Temporal validation skipped: the three expected quarters are not all present.")


**Interpretation:** compare the temporal holdout score with the household-grouped test score. A large drop may indicate that labour-market patterns or data collection conditions changed between quarters. With only three quarters, temporal evidence is limited and should not be overgeneralised.


## 6.17 How much could the excluded records change the headline shares?

**What this cell does:** first shows what kind of work evidence the 2,039 excluded records contain (own or household business questions, farming questions, temporary absence). Then it recalculates the weighted class shares under extreme assumptions: if all of them were underemployed, adequately employed, or unemployed.

**How to read the output:** this is a robustness check on the excluded group (it includes the household-use farmers the proposal mentions). If the shares barely move between the scenarios, the excluded records do not change the main message. The last scenario is an extreme case, included only to show the limit.

In [ ]:
# STEP 37X: HOW MUCH COULD THE EXCLUDED SCREENING-EVIDENCE RECORDS CHANGE THE SHARES?

flag = youth_data["employment_screening_unresolved"].fillna(False).astype(bool)

def said_yes(column):
    return youth_data[column].astype("string").str.strip().str.upper().eq("YES").fillna(False)

direct_work = said_yes("atw1") | said_yes("atw2") | said_yes("atw3")
farm_work = said_yes("agf1b_1") | said_yes("agf1b_2") | said_yes("agf1b_3") | said_yes("agf1b_4")
absent_work = said_yes("abs1a") | said_yes("abs1b")

evidence = pd.Series("Not flagged", index=youth_data.index, dtype="object")
evidence[flag & direct_work & ~farm_work & ~absent_work] = "Own or household business questions only"
evidence[flag & farm_work & ~direct_work & ~absent_work] = "Farming questions only"
evidence[flag & absent_work & ~direct_work & ~farm_work] = "Temporary-absence questions only"
evidence[flag & ((direct_work.astype(int) + farm_work.astype(int) + absent_work.astype(int)) >= 2)] = "More than one type"
evidence[flag & (evidence == "Not flagged")] = "Other"

print("WHAT KIND OF WORK EVIDENCE DO THE EXCLUDED RECORDS SHOW?")
display(pd.crosstab(evidence[flag], youth_data.loc[flag, "quarter"], margins=True))

base = model_data[["quarter", "popw", "employment_class"]].copy()
extra = youth_data.loc[flag & (pd.to_numeric(youth_data["popw"], errors="coerce") > 0), ["quarter", "popw"]].copy()

def weighted_class_shares(df):
    d = df.copy()
    d["_w"] = normalised_weights(d)
    d = d.loc[d["_w"].notna()]
    return (d.groupby("employment_class")["_w"].sum() / d["_w"].sum() * 100).reindex([0, 1, 2]).fillna(0)

scenarios = {"Current model sample (flagged records left out)": weighted_class_shares(base)}
for code, text in [(1, "If ALL flagged records were underemployed"),
                   (2, "If ALL flagged records were adequately employed"),
                   (0, "If ALL flagged records were unemployed (extreme case)")]:
    scenarios[text] = weighted_class_shares(pd.concat([base, extra.assign(employment_class=code)]))

bounds_table = pd.DataFrame(scenarios, index=[0, 1, 2]).T.round(1)
bounds_table.columns = ["Unemployed (%)", "Underemployed (%)", "Adequately employed (%)"]
print(f"\nFlagged records added in each scenario: {len(extra):,}")
display(bounds_table)


## 6.18 Save the cleaned modelling data, model comparison and trained model

**What this cell does:** exports the modelling sample, model comparison table and selected model to files in the Colab session. These outputs can be downloaded from the Files panel. Do not publish the raw NBS microdata or identifiable household/interview identifiers.


In [ ]:
# STEP 37M: EXPORT PROJECT OUTPUTS

import joblib

# Save only the analysis fields and target; omit direct identifiers.
export_columns = list(dict.fromkeys(feature_columns + ["employment_class", "quarter", "popw"]))
export_data = model_data[export_columns].copy()
export_data.to_csv("youth_labour_model_sample.csv", index=False)
results_df.to_csv("model_comparison.csv", index=False)
joblib.dump(best_model, "youth_labour_underutilisation_model.joblib")

saved = ["youth_labour_model_sample.csv", "model_comparison.csv", "youth_labour_underutilisation_model.joblib"]

# Optional tables, saved only if the earlier cells were run.
for variable_name, file_name in [("permutation_df", "permutation_importance.csv"),
                                 ("odds_table", "odds_ratios.csv"),
                                 ("sensitivity_table", "low_pay_sensitivity.csv"),
                                 ("bounds_table", "excluded_records_scenarios.csv")]:
    if variable_name in globals():
        globals()[variable_name].to_csv(file_name)
        saved.append(file_name)

print("Saved files:")
for name in saved:
    print("-", name)


## 6.19 Plain-English summary of this run

**What this cell does:** reads the results already in memory and prints a summary in words. Every number comes from the cells you ran, so you can copy it into the report without typing scores by hand.

In [ ]:
# STEP 37S: PLAIN-ENGLISH SUMMARY BUILT FROM THIS RUN'S ACTUAL NUMBERS

print("RESULTS SUMMARY (every number below comes from the cells you just ran)")
print("=" * 70)

baseline_row = results_df.loc[results_df["Model"] == "Majority baseline"].iloc[0]
best_row = results_df.loc[results_df["Model"] != "Majority baseline"].iloc[0]

print(f"\n1. Best model (excluding the baseline), chosen by macro-F1: {best_row['Model']}")
print(f"   Macro-F1: {best_row['Macro F1']:.3f}   (majority baseline: {baseline_row['Macro F1']:.3f})")
print(f"   Balanced accuracy: {best_row['Balanced accuracy']:.3f}   (baseline: {baseline_row['Balanced accuracy']:.3f})")
print(f"   Plain accuracy: {best_row['Accuracy']:.3f}   (baseline: {baseline_row['Accuracy']:.3f}; "
      "the baseline looks good on accuracy only because it always guesses the biggest class)")
print("   Recall (share of each true class that the model found):")
for label in ["Unemployed", "Underemployed", "Adequately employed"]:
    print(f"     {label}: {best_row['Recall: ' + label]:.3f}")

difference = best_row["Macro F1"] - baseline_row["Macro F1"]
print("\n2. Macro-F1 difference between the best model and the majority baseline:", f"{difference:+.3f}")
print("   The baseline predicts only one class, so its macro-F1 is always low and beating it is easy.")
print("   A more useful question is how well each class is found (the recall figures above).")
recalls_best = {label: best_row["Recall: " + label] for label in ["Unemployed", "Underemployed", "Adequately employed"]}
weakest = min(recalls_best, key=recalls_best.get)
print(f"   The class the best model finds least often is: {weakest} (recall {recalls_best[weakest]:.3f}).")

if "temporal_macro_f1" in globals():
    print(f"\n3. Later-quarter test (train Q3 2024 and Q1 2025, test Q2 2025): macro-F1 {temporal_macro_f1:.3f}"
          f" (baseline {temporal_baseline_f1:.3f}).")

if "permutation_df" in globals():
    top = permutation_df.head(3)["Predictor"].tolist()
    print("\n4. Predictors the model relies on most (permutation importance):", ", ".join(top))

if "odds_table" in globals():
    clear = odds_table.loc[(odds_table["Clearly different from 1?"] == "Yes")
                           & ~odds_table.index.str.startswith(("State: ", "Survey round"))]
    print("\n5. Factors clearly linked to underemployment, after accounting for state:")
    for name, row in clear.sort_values("Odds ratio", ascending=False).iterrows():
        direction = "higher" if row["Odds ratio"] > 1 else "lower"
        print(f"   {name}: odds ratio {row['Odds ratio']:.2f} ({direction} odds of being underemployed)")

print("\nReminder: these are associations in survey data, not proof of cause.")


# 7. Findings

The numbers below come from the outputs earlier in this notebook. Percentages marked *unweighted* count respondents in the sample. Percentages marked *weighted* use the survey weights. Model scores are printed in section 6.19 and are not repeated here, so that no number is copied by hand.

**Who is in the study.** Three NLFS quarters (Q3 2024, Q1 2025, Q2 2025) give 38,792 young people aged 15 to 34. Of these, 26,142 have a recorded job and 1,941 are unemployed (no job, searching, available). The 10,572 who are outside the labour force and 137 who were searching but not available are left out. In total 27,839 young people receive one of the three class labels.

**Finding 1: underemployment is the largest group, not unemployment** (research question 1).
- Unweighted: underemployed 15,265 (54.8%), adequately employed 10,633 (38.2%), unemployed 1,941 (7.0%).
- Weighted, by quarter: underemployed about 55.5–55.8% in all three quarters, adequately employed about 36–37%, unemployed 7.3% (Q3 2024), 8.1% (Q1 2025) and 8.3% (Q2 2025). The small rise in unemployment is not shown to be statistically meaningful.
- The official unemployment rate (4.3% for Q2 2024) therefore says little about the quality of youth work.

**Finding 2: low pay and no pay matter more than short hours** (research question 2). Among the 26,142 employed youth (unweighted): low pay 27.3%, no payment 26.4%, too few hours with a wish and availability for more 14.2%. Many people meet more than one test: 41.6% meet none, 48.9% meet one and 9.5% meet two. Counting hours only, as the official time-related measure does, therefore misses most underemployed youth.

**Finding 3: who is more affected** (research question 1; unweighted sample shares).
- *Sex:* women 58.1% underemployed against 51.2% for men; unemployment 8.0% for women and 5.8% for men.
- *Age:* underemployment falls steadily with age, from 80.2% at ages 15–19 to 37.3% at ages 30–34. Unemployment is highest at ages 20–24 (10.2%).
- *Education:* higher education goes with less underemployment but more unemployment. For example, unemployment is 24.9% among 400-level university students and graduates, 18.8% among holders of higher degrees and 17.2% among HND holders, compared with 8.4% for SS3 and 4.1% for Quranic education. Underemployment is 26.9% at 400-level university against 46.2% at SS3.
- *State:* the highest underemployment shares are in Niger (77.4%), Adamawa (76.7%), Plateau (71.9%), Katsina (70.2%) and Ebonyi (69.0%); the lowest are in Enugu (22.7%), Imo (25.5%), Lagos (27.7%), Ogun (29.7%) and Gombe (31.5%). Some of the low-underemployment states have high unemployment, so both figures are needed. State figures come from the sample and are not official state rates.
- The weighted tables with margins of error, including urban and rural sector, are printed in section 5 and are the figures to quote in the report.

**Finding 4: the data has a screening problem that must be stated.** 2,039 respondents say YES to a work-screening question but have no main-job arrangement. Of the 1,941 respondents labelled unemployed, 481 (24.8%) are in this group, which leaves 1,460 unemployed in the model sample (27,358 respondents in total). The robustness cell in section 6.17 shows how far the shares could move if these records were classified differently.

**Finding 5: how predictable is a young person's class?** (research question 4). Run sections 6.8 to 6.19 and read the printed summary. Background facts alone are expected to predict the class only partly, because many labour outcomes depend on things the survey background variables do not capture.

# 8. Key actions and recommendations

These recommendations follow from the associations found above. They are suggestions for targeting, not proof that an action will work.

1. **Measure pay and hours, not only jobs.** Report a measure that includes no pay and low pay next to the official unemployment rate. Finding 2 shows that hours alone miss most underemployed youth.
2. **Target programmes at the groups with the highest underemployment:** teenagers (ages 15–19), women, and the states with the highest shares. Check the weighted tables before choosing states, because small samples are unstable.
3. **Help unpaid family workers move into paid work.** A large share of underemployment is unpaid work in household businesses and farms. Training and credit schemes that only reach people who already have paid jobs will miss them.
4. **Do not treat education as a complete answer.** Higher education goes with less underemployment but more unemployment, so graduate job creation matters as well as skills training.
5. **Use the model only for exploration.** Background facts predict the class only partly. The model should help to see where vulnerability is concentrated; it should not be used to decide about individuals.

# 9. Limitations and next steps

**Limitations (state these clearly in the report and the defence)**
- **Only three of four quarters.** Q4 2024 was excluded because its Stata file could not be loaded and is about 40% smaller than the others. We do not claim full-year coverage and cannot account for seasonality.
- **Project-defined classes.** The underemployed class combines a time shortfall, no payment and low pay. It is **not** the official NBS time-related underemployment rate. The NBS definition counts only people who work fewer than 40 hours, want more and are available.
- **The low-pay line is a research choice.** It is two-thirds of the quarterly median, not a minimum wage or poverty line. Section 5 tests other lines.
- **Earnings are approximate.** Earnings come in bands and pay periods, so monthly values are estimates. 305 extreme or implausible values were set aside, and 244 workers could not be classified because of this.
- **Hours are usual hours**, not hours worked last week. Workers with no recorded hours cannot meet the hours rule.
- **Excluded records.** 2,039 respondents with work-screening evidence but no main-job arrangement are left out of the model; 10,572 people outside the labour force and 137 who searched but were not available are not classified. Our shares describe the *classified* youth and are not directly comparable with NBS headline rates.
- **Weights and survey design.** Weights are used, but the survey's clustering and stratification are only partly modelled, so margins of error may be too narrow.
- **Associations, not causes.** Odds ratios and model importance show links, not effects.
- **Limited predictors.** Marital status, disability and household structure were not loaded, and no infrastructure variable such as electricity is available in the individual-level files.
- **Confidentiality.** The NBS microdata are used under NBS terms. Do not share raw respondent-level files or identifiers.

**Next steps**
- Load marital status, disability and household-structure columns and test whether they improve prediction.
- Settle the 2,039 screening-evidence records by checking the questionnaire routing with the NBS documentation.
- Obtain a readable Q4 2024 file to cover a full year.
- Link state-level or household-level infrastructure data (for example electricity access).
- Fit survey-design-aware models (with clusters and strata) to get correct standard errors.

# 10. References

1. National Bureau of Statistics (2024). *Nigeria Labour Force Statistics Report, Q2 2024.* https://nigerianstat.gov.ng/download/1241594
2. AllAfrica (2024). *Nigeria's unemployment rate drops to 4.3 per cent in Q2 2024*, reporting the NBS Q2 2024 labour force report. https://allafrica.com/stories/202411260053.html
3. National Bureau of Statistics. *Nigeria Labour Force Survey report covering Q3 2023* (wage employment rate table). https://nigerianstat.gov.ng/download/1241455
4. National Bureau of Statistics. *Microdata Catalog: Nigeria Labour Force Survey* (questionnaire, instruction manual and data files). https://microdata.nigerianstat.gov.ng
5. International Labour Organization (2013). *Resolution concerning statistics of work, employment and labour underutilization*, 19th International Conference of Labour Statisticians.
6. National Bureau of Statistics. *NLFS FAQ and key definitions.* https://nigerianstat.gov.ng/page/nlfs-faq
7. National Bureau of Statistics. *Microdata catalogue, NLFS Q2 2024 to Q2 2025.* https://microdata.nigerianstat.gov.ng/index.php/catalog/152
8. National Bureau of Statistics. *Annual Nigeria Labour Force Survey report.* https://nigerianstat.gov.ng/pdfuploads/Annual_Nigerian_Labour_Force_Survey_Report.pdf